## Web search

In [1]:
import os
import json
import time
import random
import re
import requests
import webbrowser
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from serpapi import GoogleSearch

# SerpAPI Configuration
GoogleSearch.SERP_API_KEY = "45c9f68dd058e937b082a171952a097f1fe9c85f90dbe5fb4ae60b031bd872a8"

# Paths Configuration
BASE_FOLDER = '/Users/shengfang/Desktop/TRI/Rb3BiI6'
PDF_FOLDER = os.path.join(BASE_FOLDER, 'pdfs')
os.makedirs(BASE_FOLDER, exist_ok=True)
os.makedirs(PDF_FOLDER, exist_ok=True)

In [3]:
# Search for Research Papers using SerpAPI
def search_papers(query, num_results=10):
    """Search for academic papers and extract links"""
    search = GoogleSearch({
        "q": query,
        "engine": "google_scholar",
        "api_key": GoogleSearch.SERP_API_KEY,
        "num": num_results,
    })
    
    result = search.get_dict()
    
    # Save search results
    json_file_path = os.path.join(BASE_FOLDER, f'{query.replace(" ", "_")}_search_results.json')
    with open(json_file_path, 'w') as json_file:
        json.dump(result, json_file, indent=4)
    
    # Extract links
    links = []
    for item in result.get('organic_results', []):
        link = item.get('link')
        if link:
            links.append(link)
    
    # Save links
    csv_file_path = os.path.join(BASE_FOLDER, f'{query.replace(" ", "_")}_links.csv')
    with open(csv_file_path, 'w') as csv_file:
        for link in links:
            csv_file.write(link + '\n')
    
    print(f" Search completed for: {query}")
    print(f" Found {len(links)} research paper links")
    print(f" Results saved to: {json_file_path}")
    print(f" Links saved to: {csv_file_path}")
    
    return links, result

# Using multiple search terms increases coverage and quality of results

SEARCH_QUERIES = [
    # Primary search with full chemical name
    "rubidium bismuth iodide spin coating thin films",

    # Abbreviation search
    "Rb3BiI6 spin coating thin films",

    # Alternative with chemical formula and full name
    "Rb3BiI6 rubidium bismuth iodide film fabrication",

    # Perovskite context (important for this material class)
    "Rb3BiI6 rubidium bismuth iodide perovskite spin coating",

]

# Execute multiple searches for comprehensive coverage
all_links = []
all_search_results = []

for i, query in enumerate(SEARCH_QUERIES, 1):
    print(f"\n{'='*60}")
    print(f"SEARCH {i}/{len(SEARCH_QUERIES)}: {query}")
    print(f"{'='*60}")
    
    links, search_results = search_papers(query, num_results=8) 
    all_links.extend(links)
    all_search_results.append({
        'query': query,
        'results': search_results,
        'links_found': len(links)
    })
    
    # Brief pause between searches to be respectful to the API
    if i < len(SEARCH_QUERIES):
        time.sleep(2)

# Remove duplicates while preserving order
seen = set()
unique_links = []
for link in all_links:
    if link not in seen:
        seen.add(link)
        unique_links.append(link)

links = unique_links

print(f"\n{'='*60}")
print(f"COMPREHENSIVE SEARCH SUMMARY")
print(f"{'='*60}")
print(f"Total searches performed: {len(SEARCH_QUERIES)}")
print(f"=Total links found: {len(all_links)}")
print(f"Unique links after deduplication: {len(links)}")
print(f"\nSearch breakdown:")
for result in all_search_results:
    print(f"  • '{result['query'][:50]}...': {result['links_found']} links")

# Save comprehensive results
comprehensive_results = {
    'search_strategy': 'Multi-query comprehensive search for FAPbI3',
    'chemical_names': [
        'FAPbI3 (abbreviation)',
        'Formamidinium Lead Iodide (IUPAC name)',
        'HC(NH2)2PbI3 (chemical formula)',
        'Formamidinium Lead Triiodide'
    ],
    'total_queries': len(SEARCH_QUERIES),
    'queries_used': SEARCH_QUERIES,
    'total_links_found': len(all_links),
    'unique_links': len(links),
    'individual_results': all_search_results
}

comprehensive_results_file = os.path.join(BASE_FOLDER, 'comprehensive_FAPbI3_search_results.json')
with open(comprehensive_results_file, 'w') as f:
    json.dump(comprehensive_results, f, indent=4)

print(f"Comprehensive results saved to: {comprehensive_results_file}")


print(f"\n Found Links:")
for i, link in enumerate(links, 1):
    print(f"  {i}. {link}")


SEARCH 1/4: rubidium bismuth iodide spin coating thin films
 Search completed for: rubidium bismuth iodide spin coating thin films
 Found 8 research paper links
 Results saved to: /Users/shengfang/Desktop/TRI/test_FAPbI3/rubidium_bismuth_iodide_spin_coating_thin_films_search_results.json
 Links saved to: /Users/shengfang/Desktop/TRI/test_FAPbI3/rubidium_bismuth_iodide_spin_coating_thin_films_links.csv

SEARCH 2/4: Rb3BiI6 spin coating thin films
 Search completed for: Rb3BiI6 spin coating thin films
 Found 0 research paper links
 Results saved to: /Users/shengfang/Desktop/TRI/test_FAPbI3/Rb3BiI6_spin_coating_thin_films_search_results.json
 Links saved to: /Users/shengfang/Desktop/TRI/test_FAPbI3/Rb3BiI6_spin_coating_thin_films_links.csv

SEARCH 3/4: Rb3BiI6 rubidium bismuth iodide film fabrication
 Search completed for: Rb3BiI6 rubidium bismuth iodide film fabrication
 Found 2 research paper links
 Results saved to: /Users/shengfang/Desktop/TRI/test_FAPbI3/Rb3BiI6_rubidium_bismuth_iod

In [5]:
# Enhanced Institutional PDF Downloader - Performance Optimized
class InstitutionalPDFDownloader:
    """
    Main PDF downloader with institutional access and anti-detection features
    
    Performance Optimizations:
    - Reduced timeouts from 15-30s to 8-15s
    - Removed duplicate Optica handling code
    - Limited PDF candidate attempts to first 3 per URL
    - Faster filename generation and error message truncation
    - Quick PDF validation without full content inspection
    """
    
    def __init__(self):
        self.session = requests.Session()
        self.setup_session()
        self.stats = {
            'downloaded': 0,
            'failed': 0,
            'semi_automated': 0
        }
    
    def setup_session(self):
        """Setup realistic browser session to avoid bot detection"""
        user_agents = [
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2.1 Safari/605.1.15'
        ]
        
        selected_ua = random.choice(user_agents)
        
        self.session.headers.update({
            'User-Agent': selected_ua,
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.9',
            'Accept-Encoding': 'gzip, deflate, br',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1',
            'Sec-Fetch-Dest': 'document',
            'Sec-Fetch-Mode': 'navigate',
            'Sec-Fetch-Site': 'none',
            'Cache-Control': 'max-age=0'
        })
    
    def construct_pdf_urls(self, url):
        """Construct direct PDF URLs based on publisher patterns - Enhanced with robust fallbacks"""
        pdf_candidates = []
        domain = urlparse(url).netloc.lower()
        
        print(f"    Analyzing domain: {domain}")
        
        # ===== ENHANCED ARXIV HANDLING =====
        if 'arxiv.org' in domain:
            if '/abs/' in url:
                # Direct conversion from abs to pdf
                pdf_url = url.replace('/abs/', '/pdf/') + '.pdf'
                pdf_candidates.append(('constructed', pdf_url, 'arXiv PDF'))
                print(f"    ✅ Constructed arXiv PDF: {pdf_url}")
            else:
                # Fallback pattern for other arXiv URLs
                arxiv_match = re.search(r'arxiv\.org/.*?(\d+\.\d+)', url)
                if arxiv_match:
                    paper_id = arxiv_match.group(1)
                    pdf_url = f"https://arxiv.org/pdf/{paper_id}.pdf"
                    pdf_candidates.append(('constructed', pdf_url, f'arXiv PDF ({paper_id})'))
                    print(f"    ✅ Constructed arXiv PDF from ID: {pdf_url}")
        
        # ===== ENHANCED WILEY SUPPORT =====
        elif ('wiley.com' in domain or 'chemistry-europe' in domain) and '/doi/' in url:
            # Multiple Wiley patterns for better success rate
            base_url = url.split('/doi/')[0] if '/doi/' in url else url
            doi_part = url.split('/doi/')[1] if '/doi/' in url else ''
            
            wiley_patterns = []
            if '/doi/full/' in url:
                wiley_patterns.append(url.replace('/doi/full/', '/doi/pdf/'))
            elif '/doi/abs/' in url:
                wiley_patterns.append(url.replace('/doi/abs/', '/doi/pdf/'))
            elif doi_part:
                wiley_patterns.extend([
                    f"{base_url}/doi/pdf/{doi_part}",
                    f"{base_url}/doi/pdfdirect/{doi_part}"
                ])
            
            for pattern in wiley_patterns:
                pdf_candidates.append(('constructed', pattern, 'Wiley PDF'))
            print(f"    ✅ Constructed {len(wiley_patterns)} Wiley PDF patterns")
            
        # ===== ENHANCED ACS SUPPORT =====
        elif 'acs.org' in domain and '/doi/' in url:
            base_url = url.split('/doi/')[0]
            doi_part = url.split('/doi/')[1]
            acs_patterns = [
                f"{base_url}/doi/pdf/{doi_part}",
                f"{base_url}/doi/pdfplus/{doi_part}"
            ]
            for pattern in acs_patterns:
                pdf_candidates.append(('constructed', pattern, 'ACS PDF'))
            print(f"    ✅ Constructed {len(acs_patterns)} ACS PDF patterns")
            
        # ===== ENHANCED SCIENCEDIRECT SUPPORT =====
        elif 'sciencedirect.com' in domain:
            # Enhanced ScienceDirect: extract PII and try multiple patterns
            if '/pii/' in url:
                pii_match = re.search(r'/pii/([A-Z0-9]+)', url)
                if pii_match:
                    pii = pii_match.group(1)
                    # Multiple PDF URL patterns for better success rate
                    sciencedirect_patterns = [
                        f"https://www.sciencedirect.com/science/article/pii/{pii}/pdfft?md5=b&pid=1-s2.0-{pii}-main.pdf",
                        f"https://www.sciencedirect.com/science/article/pii/{pii}/pdf",
                        f"https://pdf.sciencedirectassets.com/science/article/pii/{pii}/1-s2.0-{pii}-main.pdf"
                    ]
                    for i, pattern in enumerate(sciencedirect_patterns, 1):
                        pdf_candidates.append(('constructed', pattern, f'ScienceDirect PDF Pattern {i} (PII: {pii})'))
                    print(f"    ✅ Constructed {len(sciencedirect_patterns)} ScienceDirect PDF patterns")
                    
        # ===== ENHANCED NATURE SUPPORT =====
        elif 'nature.com' in domain and '/articles/' in url:
            # Nature: add .pdf extension
            pdf_url = url.rstrip('/') + '.pdf'
            pdf_candidates.append(('constructed', pdf_url, 'Nature PDF'))
            print(f"    ✅ Constructed Nature PDF: {pdf_url}")
            
        # ===== ENHANCED PMC SUPPORT =====
        elif 'pmc.ncbi.nlm.nih.gov' in domain or 'pubmed' in domain:
            # PubMed Central - enhanced with multiple patterns
            try:
                # Try direct construction first (faster)
                pmc_match = re.search(r'PMC(\d+)', url)
                if pmc_match:
                    pmc_id = pmc_match.group(1)
                    pmc_patterns = [
                        f"https://www.ncbi.nlm.nih.gov/pmc/articles/PMC{pmc_id}/pdf/",
                        f"https://www.ncbi.nlm.nih.gov/pmc/articles/PMC{pmc_id}/pdf/main.pdf"
                    ]
                    for pattern in pmc_patterns:
                        pdf_candidates.append(('constructed', pattern, f'PMC PDF (PMC{pmc_id})'))
                    print(f"    ✅ Constructed {len(pmc_patterns)} PMC PDF patterns")
                else:
                    # Fallback to parsing if no PMC ID found
                    response = self.session.get(url, timeout=8)  # Reduced timeout
                    if response.status_code == 200:
                        soup = BeautifulSoup(response.content, 'html.parser')
                        for a_tag in soup.find_all('a', href=True):
                            href = a_tag.get('href', '')
                            text = a_tag.get_text(strip=True).lower()
                            if 'pdf' in text and ('download' in text or 'view' in text):
                                full_url = href if href.startswith('http') else urljoin(url, href)
                                pdf_candidates.append(('parsed', full_url, 'PMC PDF'))
                        print(f"    ✅ Found {len(pdf_candidates)} PMC PDF candidates")
            except Exception as e:
                print(f"    ⚠️ PMC failed: {str(e)[:30]}...")
                
        # ===== ENHANCED RSC SUPPORT =====
        elif 'rsc.org' in domain:
            # RSC: multiple pattern attempts
            if 'articlelanding' in url:
                # Direct pattern replacement
                pdf_url = url.replace('articlelanding', 'articlepdf')
                pdf_candidates.append(('constructed', pdf_url, 'RSC PDF'))
                print(f"    ✅ Constructed RSC PDF pattern")
                
            # Try HTML parsing for supplementary PDFs
            try:
                response = self.session.get(url, timeout=8)  # Reduced timeout
                if response.status_code == 200:
                    soup = BeautifulSoup(response.content, 'html.parser')
                    for a_tag in soup.find_all('a', href=True):
                        href = a_tag.get('href', '')
                        if href and (href.lower().endswith('.pdf') or '/suppdata/' in href):
                            full_url = href if href.startswith('http') else urljoin(url, href)
                            pdf_candidates.append(('parsed', full_url, 'RSC PDF'))
                    print(f"    ✅ Found {len(pdf_candidates)} RSC PDF links")
            except Exception as e:
                print(f"    ⚠️ RSC parsing failed: {str(e)[:30]}...")
                
        # ===== ENHANCED SPRINGER SUPPORT =====
        elif 'springer' in domain and '/doi/' in url:
            base_url = url.split('/doi/')[0]
            doi_part = url.split('/doi/')[1]
            springer_patterns = [
                f"{base_url}/doi/pdf/{doi_part}",
                f"{base_url}/article/{doi_part}/pdf"
            ]
            for pattern in springer_patterns:
                pdf_candidates.append(('constructed', pattern, 'Springer PDF'))
            print(f"    ✅ Constructed {len(springer_patterns)} Springer PDF patterns")
            
        # ===== ENHANCED OPTICA SUPPORT =====
        elif 'optica.org' in domain or 'opg.optica.org' in domain:
            # Optica (OSA) journals - enhanced handling
            try:
                response = self.session.get(url, timeout=8)  # Reduced timeout
                print(f"     Optica response status: {response.status_code}")
                
                if response.status_code == 202:
                    print(f"    ⚠️ HTTP 202 (Accepted) - Bot detection, using manual patterns")
                    # Construct PDF URLs based on the URI pattern
                    if 'abstract.cfm' in url and 'uri=' in url:
                        uri_match = re.search(r'uri=([^&]+)', url)
                        if uri_match:
                            uri = uri_match.group(1)
                            optica_patterns = [
                                f"https://opg.optica.org/viewmedia.cfm?uri={uri}&seq=0",
                                f"https://opg.optica.org/DirectPDFAccess/{uri}.pdf"
                            ]
                            for pattern in optica_patterns:
                                pdf_candidates.append(('manual', pattern, f'Optica PDF ({uri})'))
                    pdf_candidates.append(('manual', url, 'Optica - Manual access (HTTP 202)'))
                    
                elif response.status_code == 200:
                    soup = BeautifulSoup(response.content, 'html.parser')
                    # Look for PDF download links
                    for a_tag in soup.find_all('a', href=True):
                        href = a_tag.get('href', '')
                        text = a_tag.get_text(strip=True).lower()
                        if (('pdf' in text and any(word in text for word in ['download', 'full', 'view'])) or
                            href.endswith('.pdf') or 'viewmedia.cfm' in href):
                            full_url = href if href.startswith('http') else urljoin(url, href)
                            pdf_candidates.append(('parsed', full_url, f'Optica PDF'))
                    
                    # Direct PDF construction from abstract URL
                    if 'abstract.cfm' in url and 'uri=' in url:
                        uri_match = re.search(r'uri=([^&]+)', url)
                        if uri_match:
                            uri = uri_match.group(1)
                            pdf_url = f"https://opg.optica.org/viewmedia.cfm?uri={uri}&seq=0"
                            pdf_candidates.append(('constructed', pdf_url, f'Optica PDF ({uri})'))
                    
                    print(f"    ✅ Found {len(pdf_candidates)} Optica PDF candidates")
                else:
                    print(f"    ⚠️ HTTP {response.status_code} - adding as manual")
                    pdf_candidates.append(('manual', url, f'Optica - Manual access (HTTP {response.status_code})'))
                    
            except Exception as e:
                print(f"    ⚠️ Optica failed: {str(e)[:30]}...")
                pdf_candidates.append(('manual', url, 'Optica - Manual access required'))
        
        # ===== ENHANCED GENERIC PATTERN HANDLING =====
        # If no specific patterns found, try enhanced generic parsing with fallback patterns
        if not pdf_candidates:
            try:
                print(f"    Trying enhanced generic PDF parsing...")
                response = self.session.get(url, timeout=8)  # Reduced timeout
                
                # Handle special status codes
                if response.status_code == 202:
                    print(f"    ⚠️ HTTP 202 - needs authentication")
                    pdf_candidates.append(('manual', url, 'Requires authentication (HTTP 202)'))
                elif response.status_code == 403:
                    print(f"    ⚠️ HTTP 403 - institutional access required")
                    pdf_candidates.append(('manual', url, 'Institutional access required (HTTP 403)'))
                elif response.status_code == 200:
                    soup = BeautifulSoup(response.content, 'html.parser')
                    
                    # Look for PDF links (limit to first 5 for performance)
                    pdf_links_found = 0
                    for a_tag in soup.find_all('a', href=True):
                        if pdf_links_found >= 5:  # Limit for performance
                            break
                            
                        href = a_tag.get('href', '')
                        text = a_tag.get_text(strip=True).lower()
                        
                        # Check for PDF indicators
                        if (href.lower().endswith('.pdf') or 
                            'pdf' in text and any(word in text for word in ['download', 'full', 'view']) or
                            '/pdf' in href.lower()):
                            
                            full_url = href if href.startswith('http') else urljoin(url, href)
                            pdf_candidates.append(('parsed', full_url, 'Generic PDF'))
                            pdf_links_found += 1
                    
                    # ===== ENHANCED FALLBACK PATTERN CONSTRUCTION =====
                    # Try enhanced pattern construction for missed cases
                    if not pdf_candidates:
                        # DOI-based fallback patterns
                        if '/doi/' in url:
                            base_url = url.split('/doi/')[0]
                            doi_part = url.split('/doi/')[1]
                            fallback_patterns = [
                                f"{base_url}/doi/pdf/{doi_part}",
                                f"{base_url}/doi/pdfplus/{doi_part}",
                                f"{base_url}/doi/pdfdirect/{doi_part}"
                            ]
                            for pattern in fallback_patterns:
                                pdf_candidates.append(('constructed', pattern, 'DOI Fallback PDF'))
                            print(f"    ✅ Added {len(fallback_patterns)} DOI fallback patterns")
                        
                        # ArXiv fallback for missed cases
                        elif 'arxiv' in domain:
                            arxiv_match = re.search(r'(\d+\.\d+)', url)
                            if arxiv_match:
                                paper_id = arxiv_match.group(1)
                                pdf_url = f"https://arxiv.org/pdf/{paper_id}.pdf"
                                pdf_candidates.append(('constructed', pdf_url, f'arXiv Fallback ({paper_id})'))
                                print(f"    ✅ Added arXiv fallback pattern")
                    
                    if pdf_candidates:
                        print(f"    ✅ Found {len(pdf_candidates)} enhanced PDF candidates")
                    else:
                        print(f"     No PDF links found in HTML")
                        # Add the original URL as a manual candidate
                        pdf_candidates.append(('manual', url, 'Manual download required'))
                else:
                    print(f" HTTP {response.status_code} - adding as manual download")
                    pdf_candidates.append(('manual', url, f'Manual download required (HTTP {response.status_code})'))
                        
            except Exception as e:
                print(f"Enhanced parsing failed: {str(e)}")
                # Add as manual download candidate
                pdf_candidates.append(('manual', url, f'Manual download required (parsing failed)'))
        
        return pdf_candidates
    
    def download_pdf(self, pdf_url, save_path, original_url=None):
        """Download PDF with institutional access support - optimized for speed"""
        try:
            print(f" Downloading: {pdf_url[:60]}...")
            
            # Set proper referrer if provided
            if original_url:
                self.session.headers.update({'Referer': original_url})
            
            response = self.session.get(pdf_url, timeout=15, allow_redirects=True)  # Reduced timeout
            
            print(f" Status: {response.status_code}")
            
            if response.status_code == 200:
                content = response.content
                
                # Quick PDF validation
                if len(content) > 1000 and content.startswith(b'%PDF-'):
                    with open(save_path, 'wb') as f:
                        f.write(content)
                    file_size = os.path.getsize(save_path)
                    self.stats['downloaded'] += 1
                    return True, f"Downloaded ({file_size//1024} KB)"
                
                # Check content type
                content_type = response.headers.get('Content-Type', '').lower()
                if 'application/pdf' in content_type and len(content) > 1000:
                    with open(save_path, 'wb') as f:
                        f.write(content)
                    file_size = os.path.getsize(save_path)
                    self.stats['downloaded'] += 1
                    return True, f"Downloaded ({file_size//1024} KB)"
                
                # Quick HTML check
                if b'<html' in content[:500].lower():
                    return False, "Institutional auth required (HTML page)"
                else:
                    return False, "Not a valid PDF"
            
            elif response.status_code == 403:
                return False, "Access forbidden - institutional auth required"
            else:
                return False, f"HTTP {response.status_code}"
                
        except Exception as e:
            return False, str(e)[:30] + "..."
    
    def open_in_browser(self, pdf_url, title="PDF"):
        """Open PDF URL in browser for semi-automated download"""
        try:
            print(f"Opening {title} in browser...")
            webbrowser.open(pdf_url)
            self.stats['semi_automated'] += 1
            return True, "Opened in browser"
        except Exception as e:
            return False, str(e)

print("✅ InstitutionalPDFDownloader class created")

✅ InstitutionalPDFDownloader class created


In [7]:
def download_pdfs_from_links(links, pdf_folder):
    """
    Enhanced main function to download PDFs from research links with intelligent fallback
    Incorporates robust patterns and better error handling
    
    Returns:
        tuple: (downloader, downloaded_pdfs, failed_downloads, semi_automated_pdfs)
    """
    downloader = InstitutionalPDFDownloader()
    
    downloaded_pdfs = []
    failed_downloads = []
    semi_automated_pdfs = []
    
    print(f"🚀 STARTING ENHANCED PDF DOWNLOAD PROCESS")
    print("=" * 60)
    
    for i, link in enumerate(links, 1):
        print(f"\n[{i}/{len(links)}] Processing: {link}")
        
        try:
            # Get PDF candidates using enhanced publisher-specific patterns
            pdf_candidates = downloader.construct_pdf_urls(link)
            
            if not pdf_candidates:
                print(f"    ⚠️  No PDF candidates found")
                failed_downloads.append({"url": link, "error": "No PDF candidates found", "category": "no_candidates"})
                continue

            print(f"    Found {len(pdf_candidates)} PDF candidates")

            # ===== ENHANCED CANDIDATE PRIORITIZATION =====
            # Sort candidates by priority: constructed > parsed > manual
            priority_order = {'constructed': 1, 'parsed': 2, 'manual': 3}
            pdf_candidates.sort(key=lambda x: priority_order.get(x[0], 4))
            
            # Try downloading each PDF candidate with intelligent limits
            pdf_downloaded = False
            best_manual_candidate = None
            download_attempts = 0
            max_attempts = min(5, len(pdf_candidates))  # Try up to 5 candidates
            
            for method, pdf_url, description in pdf_candidates[:max_attempts]:
                if pdf_downloaded:
                    break
                
                download_attempts += 1
                print(f"    [{download_attempts}/{max_attempts}] Trying: {description}")
                
                # Generate clean filename with better naming
                parsed = urlparse(link)
                domain_clean = parsed.netloc.replace('www.', '').replace('.', '_')
                path_clean = parsed.path.replace('/', '_').strip('_')[:40]
                pdf_name = f"{domain_clean}_{path_clean}_{i}.pdf"
                pdf_path = os.path.join(pdf_folder, pdf_name)
                
                if method == 'manual':
                    # Add to semi-automated list (requires browser/manual access)
                    if not best_manual_candidate:
                        best_manual_candidate = {
                            "title": description,
                            "url": link,
                            "pdf_url": pdf_url,
                            "reason": "Requires institutional authentication or manual access",
                            "domain": parsed.netloc,
                            "priority": "high" if 'arxiv' in pdf_url.lower() or 'pmc' in pdf_url.lower() else "medium"
                        }
                    print(f"        🔖 Marked for browser opening: {description}")
                else:
                    # Attempt automatic download
                    success, message = downloader.download_pdf(pdf_url, pdf_path, link)
                    
                    if success:
                        print(f"        ✅ Downloaded: {pdf_name}")
                        downloaded_pdfs.append({
                            "url": link, 
                            "pdf_path": pdf_path, 
                            "pdf_name": pdf_name,
                            "method": description,
                            "domain": parsed.netloc
                        })
                        pdf_downloaded = True
                    else:
                        print(f"        ❌ Failed: {message[:50]}...")
                        
                        # ===== ENHANCED ERROR CATEGORIZATION =====
                        # Categorize errors for better fallback decisions
                        error_lower = message.lower()
                        if any(keyword in error_lower for keyword in 
                               ["institutional", "authentication", "forbidden", "403"]):
                            # This suggests institutional access is needed
                            if not best_manual_candidate:
                                best_manual_candidate = {
                                    "title": description,
                                    "url": link,
                                    "pdf_url": pdf_url,
                                    "reason": f"Institutional access required: {message}",
                                    "domain": parsed.netloc,
                                    "priority": "high"
                                }
                        elif any(keyword in error_lower for keyword in 
                                ["html", "not a valid pdf", "too small"]):
                            # This suggests we got a login/paywall page
                            if not best_manual_candidate:
                                best_manual_candidate = {
                                    "title": description,
                                    "url": link,
                                    "pdf_url": pdf_url,
                                    "reason": f"Paywall detected: {message}",
                                    "domain": parsed.netloc,
                                    "priority": "high"
                                }
            
            # ===== ENHANCED RESULT HANDLING =====
            # If no automatic download succeeded, handle intelligently
            if not pdf_downloaded:
                if best_manual_candidate:
                    print(f"    🌐 Adding to semi-automated: {best_manual_candidate['title'][:40]}...")
                    semi_automated_pdfs.append(best_manual_candidate)
                else:
                    error_info = {
                        "url": link, 
                        "error": f"All {download_attempts} download attempts failed",
                        "category": "all_failed",
                        "domain": urlparse(link).netloc,
                        "attempts": download_attempts
                    }
                    failed_downloads.append(error_info)
                    print(f"    ❌ Complete failure after {download_attempts} attempts")
            
            # Brief pause for rate limiting
            time.sleep(0.3)  # Slightly longer pause for stability
            
        except Exception as e:
            print(f"    ❌ Error processing {link}: {str(e)[:50]}...")
            failed_downloads.append({
                "url": link, 
                "error": str(e),
                "category": "processing_error",
                "domain": urlparse(link).netloc if link else "unknown"
            })
    
    # ===== ENHANCED SUMMARY =====
    print(f"\n{'='*60}")
    print(f"📊 ENHANCED DOWNLOAD SUMMARY")
    print(f"✅ Automatically downloaded: {len(downloaded_pdfs)} PDFs")
    print(f"🌐 Semi-automated (browser): {len(semi_automated_pdfs)} PDFs")
    print(f"❌ Failed: {len(failed_downloads)} PDFs")
    
    if downloaded_pdfs:
        print(f"\n✅ SUCCESSFUL DOWNLOADS:")
        for pdf in downloaded_pdfs:
            print(f"   • {pdf['pdf_name']} ({pdf['method']})")
    
    if semi_automated_pdfs:
        print(f"\n🌐 SEMI-AUTOMATED (Manual Required):")
        high_priority = [p for p in semi_automated_pdfs if p.get('priority') == 'high']
        medium_priority = [p for p in semi_automated_pdfs if p.get('priority') != 'high']
        
        if high_priority:
            print(f"   🔴 HIGH PRIORITY ({len(high_priority)}):")
            for pdf in high_priority:
                print(f"      • {pdf['domain']} - {pdf['reason'][:50]}...")
        
        if medium_priority:
            print(f"   🟡 MEDIUM PRIORITY ({len(medium_priority)}):")
            for pdf in medium_priority:
                print(f"      • {pdf['domain']} - {pdf['reason'][:50]}...")
    
    print(f"\n📁 Saved to: {pdf_folder}")
    
    return downloader, downloaded_pdfs, failed_downloads, semi_automated_pdfs

# Execute the enhanced download process with robust error handling
print("🚀 Starting enhanced PDF download process with robust patterns...")
downloader, downloaded_pdfs, failed_downloads, semi_automated_pdfs = download_pdfs_from_links(links, PDF_FOLDER)

🚀 Starting enhanced PDF download process with robust patterns...
🚀 STARTING ENHANCED PDF DOWNLOAD PROCESS

[1/11] Processing: https://pubs.acs.org/doi/abs/10.1021/acs.chemmater.8b01341
    Analyzing domain: pubs.acs.org
    ✅ Constructed 2 ACS PDF patterns
    Found 2 PDF candidates
    [1/2] Trying: ACS PDF
 Downloading: https://pubs.acs.org/doi/pdf/abs/10.1021/acs.chemmater.8b013...
 Status: 404
        ❌ Failed: HTTP 404...
    [2/2] Trying: ACS PDF
 Downloading: https://pubs.acs.org/doi/pdfplus/abs/10.1021/acs.chemmater.8...
 Status: 404
        ❌ Failed: HTTP 404...
    ❌ Complete failure after 2 attempts

[2/11] Processing: https://pubs.rsc.org/en/content/articlehtml/2022/ra/d2ra05484a
    Analyzing domain: pubs.rsc.org
    ✅ Found 0 RSC PDF links
    Trying enhanced generic PDF parsing...
    ✅ Found 1 enhanced PDF candidates
    Found 1 PDF candidates
    [1/1] Trying: Generic PDF
 Downloading: https://pubs.rsc.org/en/content/articlepdf/2022/ra/d2ra05484...
 Status: 200
       

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


     No PDF links found in HTML
    Found 1 PDF candidates
    [1/1] Trying: Manual download required
        🔖 Marked for browser opening: Manual download required
    🌐 Adding to semi-automated: Manual download required...

[9/11] Processing: https://nitech.repo.nii.ac.jp/record/2000415/files/ko1355_f.pdf
    Analyzing domain: nitech.repo.nii.ac.jp
    Trying enhanced generic PDF parsing...


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


     No PDF links found in HTML
    Found 1 PDF candidates
    [1/1] Trying: Manual download required
        🔖 Marked for browser opening: Manual download required
    🌐 Adding to semi-automated: Manual download required...

[10/11] Processing: https://pubs.acs.org/doi/abs/10.1021/acs.chemmater.5b03147
    Analyzing domain: pubs.acs.org
    ✅ Constructed 2 ACS PDF patterns
    Found 2 PDF candidates
    [1/2] Trying: ACS PDF
 Downloading: https://pubs.acs.org/doi/pdf/abs/10.1021/acs.chemmater.5b031...
 Status: 404
        ❌ Failed: HTTP 404...
    [2/2] Trying: ACS PDF
 Downloading: https://pubs.acs.org/doi/pdfplus/abs/10.1021/acs.chemmater.5...
 Status: 404
        ❌ Failed: HTTP 404...
    ❌ Complete failure after 2 attempts

[11/11] Processing: https://search.proquest.com/openview/e2e60b6e786d5471415089021d90a595/1?pq-origsite=gscholar&cbl=2026366&diss=y
    Analyzing domain: search.proquest.com
    Trying enhanced generic PDF parsing...
     No PDF links found in HTML
    Found 1 

In [8]:
# Enhanced Semi-Automated Download (Browser Opening)
def open_pdfs_in_browser(semi_automated_pdfs, downloader):
    """
    Enhanced function to open PDFs in browser for semi-automated download
    Prioritizes high-priority PDFs and provides better user guidance
    
    Args:
        semi_automated_pdfs: List of PDF info dictionaries
        downloader: Instance of InstitutionalPDFDownloader
    
    Returns:
        int: Number of PDFs successfully opened in browser
    """
    if not semi_automated_pdfs:
        print("🎉 No semi-automated downloads needed - all PDFs were fully automated!")
        return 0
    
    print(f"\n🌐 ENHANCED SEMI-AUTOMATED DOWNLOAD PROCESS")
    print("=" * 60)
    
    # Sort by priority (high priority first)
    high_priority = [p for p in semi_automated_pdfs if p.get('priority') == 'high']
    medium_priority = [p for p in semi_automated_pdfs if p.get('priority') != 'high']
    sorted_pdfs = high_priority + medium_priority
    
    opened_count = 0
    
    # Process high priority first
    if high_priority:
        print(f"\n🔴 HIGH PRIORITY PDFs ({len(high_priority)}) - Likely to be freely accessible:")
        for i, pdf_info in enumerate(high_priority, 1):
            print(f"\n[HIGH {i}/{len(high_priority)}] Opening: {pdf_info['title']}")
            print(f"    🌐 URL: {pdf_info['pdf_url']}")
            print(f"    📝 Domain: {pdf_info['domain']}")
            print(f"    💡 Note: {pdf_info['reason']}")
            
            # Provide specific guidance based on domain
            domain = pdf_info['domain'].lower()
            if 'arxiv' in domain:
                print(f"    ✅ arXiv - Should be freely downloadable!")
            elif 'pmc.ncbi.nlm.nih.gov' in domain:
                print(f"    ✅ PubMed Central - Open access repository")
            elif 'nature.com' in domain or 'science.org' in domain:
                print(f"    🔒 High-impact journal - May require institutional access")
            
            success, message = downloader.open_in_browser(pdf_info['pdf_url'], pdf_info['title'])
            
            if success:
                print(f"    ✅ Successfully opened in browser")
                opened_count += 1
            else:
                print(f"    ❌ Failed to open: {message}")
            
            # Brief pause between high priority opens
            if i < len(high_priority):
                print(f"    ⏳ Waiting 2 seconds before next high-priority PDF...")
                time.sleep(2)
    
    # Process medium priority
    if medium_priority:
        print(f"\n🟡 MEDIUM PRIORITY PDFs ({len(medium_priority)}) - May require institutional access:")
        for i, pdf_info in enumerate(medium_priority, 1):
            print(f"\n[MED {i}/{len(medium_priority)}] Opening: {pdf_info['title']}")
            print(f"    🌐 URL: {pdf_info['pdf_url']}")
            print(f"    📝 Domain: {pdf_info['domain']}")
            print(f"    💡 Note: {pdf_info['reason']}")
            
            # Provide domain-specific guidance
            domain = pdf_info['domain'].lower()
            if 'wiley' in domain or 'springer' in domain or 'elsevier' in domain:
                print(f"    🔒 Major publisher - Likely requires subscription")
            elif 'acs.org' in domain:
                print(f"    🔒 ACS journal - May have institutional access")
            elif 'rsc.org' in domain:
                print(f"    🔒 RSC journal - Check for open access version")
            
            success, message = downloader.open_in_browser(pdf_info['pdf_url'], pdf_info['title'])
            
            if success:
                print(f"    ✅ Successfully opened in browser")
                opened_count += 1
            else:
                print(f"    ❌ Failed to open: {message}")
            
            # Longer pause between medium priority opens
            if i < len(medium_priority):
                print(f"    ⏳ Waiting 3 seconds before next medium-priority PDF...")
                time.sleep(3)
    
    # Enhanced summary with actionable guidance
    print(f"\n📊 SEMI-AUTOMATED SUMMARY:")
    print(f"    ✅ Successfully opened in browser: {opened_count} PDFs")
    print(f"    🔴 High priority (likely free): {len(high_priority)} PDFs")
    print(f"    🟡 Medium priority (may need access): {len(medium_priority)} PDFs")
    
    if opened_count > 0:
        print(f"\n💡 DOWNLOAD GUIDANCE:")
        print(f"    1. Check each opened browser tab")
        print(f"    2. Look for 'Download PDF' or 'Full Text' buttons")
        print(f"    3. For paywalled content, try:")
        print(f"       • University VPN or campus network")
        print(f"       • Institutional library access")
        print(f"       • Open access versions on arXiv/PMC")
        print(f"       • Author's personal/institutional webpage")
        print(f"    4. Save PDFs to: {PDF_FOLDER}")
    
    return opened_count

# Execute enhanced semi-automated download if needed
if semi_automated_pdfs:
    print(f"\n" + "="*60)
    opened_count = open_pdfs_in_browser(semi_automated_pdfs, downloader)
    
    # Update validation after browser opening
    print(f"\n🔄 After browser opening, you can re-run validation:")
    print("validation_results = validate_pdf_collection(PDF_FOLDER, downloaded_pdfs, semi_automated_pdfs)")
else:
    print("\n🎉 All PDFs were downloaded automatically! No manual steps needed.")



🌐 ENHANCED SEMI-AUTOMATED DOWNLOAD PROCESS

🔴 HIGH PRIORITY PDFs (3) - Likely to be freely accessible:

[HIGH 1/3] Opening: ScienceDirect PDF Pattern 1 (PII: S0038092X21008975)
    🌐 URL: https://www.sciencedirect.com/science/article/pii/S0038092X21008975/pdfft?md5=b&pid=1-s2.0-S0038092X21008975-main.pdf
    📝 Domain: www.sciencedirect.com
    💡 Note: Institutional access required: Access forbidden - institutional auth required
Opening ScienceDirect PDF Pattern 1 (PII: S0038092X21008975) in browser...
    ✅ Successfully opened in browser
    ⏳ Waiting 2 seconds before next high-priority PDF...

[HIGH 2/3] Opening: Wiley PDF
    🌐 URL: https://onlinelibrary.wiley.com/doi/pdf/10.1002/adfm.202203300
    📝 Domain: onlinelibrary.wiley.com
    💡 Note: Institutional access required: Access forbidden - institutional auth required
Opening Wiley PDF in browser...
    ✅ Successfully opened in browser
    ⏳ Waiting 2 seconds before next high-priority PDF...

[HIGH 3/3] Opening: ScienceDirect PDF

In [ ]:
from paperqa import Settings, ask
from paperqa.settings import AgentSettings, ParsingSettings
import os
import asyncio
import glob

# Configure environment for local Ollama model usage
os.environ['OPENAI_API_KEY'] = "ollama"

# Local LLM configuration
local_llm_config = {
    'model_list': [{
        'model_name': 'ollama/llama3.2',
        'litellm_params': {
            'model': 'ollama/llama3.2',
            'api_base': "http://localhost:11434",
            'temperature': 0.1,  
            'max_tokens': 4096,  
        }
    }]
}

# Disable online API access for fully local operation
os.environ['CROSSREF_MAILTO'] = ""  # Disable Crossref metadata lookup
os.environ['SEMANTIC_SCHOLAR_API_KEY'] = ""  # Disable Semantic Scholar API
os.environ['PAPERPILE_API_KEY'] = ""  # Disable any other potential APIs

# PaperQA settings for LOCAL PDF analysis 
synthesis_settings = Settings(
    llm='ollama/llama3.2',
    llm_config=local_llm_config,
    summary_llm='ollama/llama3.2',
    summary_llm_config=local_llm_config,
    embedding='ollama/mxbai-embed-large',
    agent=AgentSettings(
        agent_llm='ollama/llama3.2', 
        agent_llm_config=local_llm_config,
        timeout=600,
        tool_names=["gen_answer", "gather_evidence"],  # Local tools only
    ),
    # NOTE: Update this path for your target composition
    paper_directory="/Users/shengfang/Desktop/TRI/Rb3BiI6/pdfs",
    parsing=ParsingSettings(
        use_doc_details=False,  # Disable metadata enrichment
        chunk_size=3000,  
        overlap=300,
    ),
    agent_type="ToolSelector",  # Force local-only analysis
)

print("PaperQA configuration complete!")
print(f"PDF directory: {synthesis_settings.paper_directory}")

# Load local PDF documents
pdf_files = glob.glob(os.path.join(synthesis_settings.paper_directory, "*.pdf"))
print(f"Found {len(pdf_files)} PDF files for analysis")

if pdf_files:
    print("\nAvailable research papers:")
    for i, pdf in enumerate(pdf_files[:5], 1):
        filename = os.path.basename(pdf)
        print(f"  {i}. {filename}")
    if len(pdf_files) > 5:
        print(f"  ... and {len(pdf_files) - 5} additional files")
else:
    print("No PDF files found in the specified directory")

PaperQA configuration complete!
PDF directory: /Users/shengfang/Desktop/TRI/test_FAPbI3/pdfs
Found 23 PDF files for analysis

Available research papers:
  1. pubs.rsc.org_en_content_articlehtml_2015_mh_c5mh00170f.pdf
  2. pubs.rsc.org_en_content_articlehtml_2019_nr_c8nr10267h.pdf
  3. document.pdf
  4. science.aan2301.pdf
  5. materials-16-01049-v3.pdf
  ... and 18 additional files


In [7]:
import asyncio
import nest_asyncio
import os
import glob
from paperqa import Docs, Settings
from paperqa.settings import ParsingSettings

# Enable async support in Jupyter
nest_asyncio.apply()
os.environ['OPENAI_API_KEY'] = "ollama"

async def local_synthesis_analysis(query, pdf_directory, analysis_name):
    """
    Perform LOCAL-ONLY PaperQA analysis for synthesis research
    
    Args:
        query: Research question to analyze (customize for your target composition)
        pdf_directory: Path to PDF collection (update for different compositions)
        analysis_name: Description for progress tracking
    
    Returns:
        PaperQA result object with answer and contexts
    """
    print(f"Starting: {analysis_name}")
    print(f"Analyzing papers in: {pdf_directory}")
    
    # Fully offline settings - no metadata enrichment
    # Local LLM configuration
    local_llm_config = {
        'model_list': [{
            'model_name': 'ollama/llama3.2',
            'litellm_params': {
                'model': 'ollama/llama3.2',
                'api_base': "http://localhost:11434",
                'temperature': 0.0,
                'max_tokens': 2048,
                'top_p': 1.0,
                'seed': 42,
                'top_k': -1,
                'repeat_penalty': 1.0
            }
        }]
    }
    
    offline_settings = Settings(
        llm='ollama/llama3.2',
        llm_config=local_llm_config,
        summary_llm='ollama/llama3.2',
        summary_llm_config=local_llm_config,
        embedding='ollama/mxbai-embed-large',
        paper_directory="/Users/shengfang/Desktop/TRI/Rb3BiI6/pdfs",
        parsing=ParsingSettings(
            use_doc_details=False,  # Critical: Disable metadata lookup
            chunk_size=2000,
            overlap=200,
        )
    )
    
    # Initialize document collection
    docs = Docs()
    pdf_files = glob.glob(os.path.join(pdf_directory, "*.pdf"))
    
    if not pdf_files:
        print(" No PDF files found")
        return None
        
    print(f"Processing {len(pdf_files)} research papers...")
    
    # Load and process PDFs with offline settings
    for i, pdf_file in enumerate(pdf_files, 1):
        filename = os.path.basename(pdf_file)
        print(f"  {i}/{len(pdf_files)} Loading: {filename[:50]}...")
        await docs.aadd(pdf_file, settings=offline_settings)
    
    # Execute analysis query
    print("Analyzing with local LLM...")
    result = await docs.aquery(query, settings=offline_settings)
    print(f"{analysis_name} completed!")
    
    return result

def run_local_analysis(query, pdf_directory, analysis_name):
    """Synchronous wrapper for modular local analysis"""
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
    
    return loop.run_until_complete(local_synthesis_analysis(query, pdf_directory, analysis_name))


In [13]:
async def qwen_local_synthesis_analysis(query, pdf_directory, analysis_name):
    """
    Perform LOCAL-ONLY PaperQA analysis for synthesis research
    
    Args:
        query: Research question to analyze (customize for your target composition)
        pdf_directory: Path to PDF collection (update for different compositions)
        analysis_name: Description for progress tracking
    
    Returns:
        PaperQA result object with answer and contexts
    """
    print(f"Starting: {analysis_name}")
    print(f"Analyzing papers in: {pdf_directory}")
    
    # Fully offline settings - no metadata enrichment
    # Local LLM configuration
    local_llm_config = {
        'model_list': [{
            'model_name': 'ollama/qwen2.5:14b',
            'litellm_params': {
                'model': 'ollama/qwen2.5:14b',
                'api_base': "http://localhost:11434",
                'temperature': 0.0,
                'max_tokens': 2048,
                'top_p': 1.0,
                'seed': 42,
                'top_k': -1,
                'repeat_penalty': 1.0,
                'timeout': 600,
            }
        }]
    }
    
    offline_settings = Settings(
        llm='ollama/qwen2.5:14b',
        llm_config=local_llm_config,
        summary_llm='ollama/qwen2.5:14b',
        summary_llm_config=local_llm_config,
        embedding='ollama/mxbai-embed-large',
        paper_directory="/Users/shengfang/Desktop/TRI/Rb3BiI6/pdfs",
        parsing=ParsingSettings(
            use_doc_details=False,  # Critical: Disable metadata lookup
            chunk_size=2000,
            overlap=200,
        )
    )
    
    # Initialize document collection
    docs = Docs()
    pdf_files = glob.glob(os.path.join(pdf_directory, "*.pdf"))
    
    if not pdf_files:
        print(" No PDF files found")
        return None
        
    print(f"Processing {len(pdf_files)} research papers...")
    
    # Load and process PDFs with offline settings
    for i, pdf_file in enumerate(pdf_files, 1):
        filename = os.path.basename(pdf_file)
        print(f"  {i}/{len(pdf_files)} Loading: {filename[:50]}...")
        await docs.aadd(pdf_file, settings=offline_settings)
    
    # Execute analysis query
    print("Analyzing with local LLM...")
    result = await docs.aquery(query, settings=offline_settings)
    print(f"{analysis_name} completed!")
    
    return result

def run_qwen_local_analysis(query, pdf_directory, analysis_name):
    """Synchronous wrapper for modular local analysis"""
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)

    return loop.run_until_complete(qwen_local_synthesis_analysis(query, pdf_directory, analysis_name))


In [16]:
TARGET_COMPOSITION = "Rb3BiI6"  # Change this for your target material
pdf_dir = "/Users/shengfang/Desktop/TRI/Rb3BiI6/pdfs"  # Update path as needed

# Focused query for direct synthesis information
direct_synthesis_query = f"""
Extract specific synthesis details for {TARGET_COMPOSITION} thin films:

PRECURSORS:
- Chemicals used (primary precursors and alternatives)
- Molar ratios and concentrations
- Purity requirements and suppliers
- Primary solvents (DMF, DMSO, others)
- Solvent ratios for mixed systems
- Dissolution conditions (temperature, time)

PROCESSING PARAMETERS:
- Spin-coating speeds and times
- Spin-coating steps (one-step vs multi-step, with or without antisolvent)
- Substrate temperatures (with or without preheating)
- Annealing temperatures and durations
- Atmosphere requirements (N2, air, vacuum)

FILM PROPERTIES:
- Film uniformity (homogeneity, defects)
- Film morphology (crystallinity, grain size)
- Film thicknesses
- Structural and optical properties

Provide quantitative values when available.
"""

In [17]:
# Execute extraction
llama_focused_result = run_local_analysis(direct_synthesis_query, pdf_dir, f"Direct {TARGET_COMPOSITION} Synthesis")

# Display results
if llama_focused_result:
    print("\n" + "="*60)
    print(f"DIRECT {TARGET_COMPOSITION} SYNTHESIS RESULTS")
    print("="*60)
    
    # Extract answer
    if hasattr(llama_focused_result, 'answer'):
        answer_text = llama_focused_result.answer
    elif hasattr(llama_focused_result, 'formatted_answer'):
        answer_text = llama_focused_result.formatted_answer
    else:
        answer_text = str(llama_focused_result)
    
    print(answer_text)
    
    # Show evidence summary
    if hasattr(llama_focused_result, 'contexts') and llama_focused_result.contexts:
        print(f"\nEvidence: {len(llama_focused_result.contexts)} relevant passages found")

        # Display example context
        print("\nExample supporting evidence:")
        try:
            first_ctx = llama_focused_result.contexts[0]
            if hasattr(first_ctx, 'text'):
                ctx_text = str(first_ctx.text)[:200] if first_ctx.text else "No text"
            else:
                ctx_text = str(first_ctx)[:200]
            print(f"   {ctx_text}...")
        except:
            print("   [Evidence available but not displayable]")
    else:
        print("\nNo direct evidence found")

    print(f"\n✅ Direct {TARGET_COMPOSITION} synthesis analysis complete!")
else:
    print(f"❌ Direct {TARGET_COMPOSITION} synthesis analysis failed")

Starting: Direct Rb3BiI6 Synthesis
Analyzing papers in: /Users/shengfang/Desktop/TRI/Rb3BiI6/pdfs
Processing 10 research papers...
  1/10 Loading: ko1280_f.pdf...


17:02:10 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:02:10 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:02:10 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:02:10 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30


  2/10 Loading: Adv Funct Materials - 2022 - Chakraborty - Rudorff...


17:02:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:02:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:02:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:02:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30


  3/10 Loading: The_Synthesis,_and_Structural_.pdf...


...

  9/10 Loading: 786773.pdf...


17:06:27 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:06:27 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:06:27 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:06:27 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
17:06:36 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:06:36 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Curre

  10/10 Loading: s11664-021-09330-8.pdf...
Analyzing with local LLM...


17:06:44 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:06:44 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:06:44 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:06:44 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/lla

Direct Rb3BiI6 Synthesis completed!

DIRECT Rb3BiI6 SYNTHESIS RESULTS
The synthesis of Rb3BiI6 thin films involves the use of primary precursors such as silver(I) iodide, bismuth(III) iodide, and bismuth(III) tris(4-methylbenzodithioate) [Bi(S2CAr)3]. The molar ratio of the precursors is not explicitly stated, but the use of a 3:1 vol mixture of DMSO:DMF with 12 vol.% HI added for improved solubility suggests a specific ratio. The solution is then spin-coated and annealed at 130°C for 15 min to form the required film.

Processing parameters include spin-coating speeds and times, with a spin speed of 3000 rpm to 6000 rpm, and spin time not specified. Substrate temperatures are not mentioned, but annealing temperatures range from 150 to 320°C. The annealing temperatures were maintained without intentional substrate heating, and the films were subjected to postdeposition annealing under BiI3 vapor.

Film properties include film uniformity, morphology, thickness, and structural and optical

In [18]:
# Execute extraction
qwen_focused_result = run_local_analysis(direct_synthesis_query, pdf_dir, f"Direct {TARGET_COMPOSITION} Synthesis")

# Display results
if qwen_focused_result:
    print("\n" + "="*60)
    print(f"DIRECT {TARGET_COMPOSITION} SYNTHESIS RESULTS")
    print("="*60)
    
    # Extract answer
    if hasattr(qwen_focused_result, 'answer'):
        answer_text = qwen_focused_result.answer
    elif hasattr(qwen_focused_result, 'formatted_answer'):
        answer_text = qwen_focused_result.formatted_answer
    else:
        answer_text = str(qwen_focused_result)
    
    print(answer_text)
    
    # Show evidence summary
    if hasattr(qwen_focused_result, 'contexts') and qwen_focused_result.contexts:
        print(f"\nEvidence: {len(qwen_focused_result.contexts)} relevant passages found")

        # Display example context
        print("\nExample supporting evidence:")
        try:
            first_ctx = qwen_focused_result.contexts[0]
            if hasattr(first_ctx, 'text'):
                ctx_text = str(first_ctx.text)[:200] if first_ctx.text else "No text"
            else:
                ctx_text = str(first_ctx)[:200]
            print(f"   {ctx_text}...")
        except:
            print("   [Evidence available but not displayable]")
    else:
        print("\nNo direct evidence found")

    print(f"\n✅ Direct {TARGET_COMPOSITION} synthesis analysis complete!")
else:
    print(f"❌ Direct {TARGET_COMPOSITION} synthesis analysis failed")

Starting: Direct Rb3BiI6 Synthesis
Analyzing papers in: /Users/shengfang/Desktop/TRI/Rb3BiI6/pdfs
Processing 10 research papers...
  1/10 Loading: ko1280_f.pdf...


17:15:35 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:15:35 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:15:35 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:15:35 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30


  2/10 Loading: Adv Funct Materials - 2022 - Chakraborty - Rudorff...


17:16:21 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:16:21 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:16:21 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:16:21 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30


  3/10 Loading: The_Synthesis,_and_Structural_.pdf...


...

  9/10 Loading: 786773.pdf...


17:19:51 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:19:51 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:19:51 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:19:51 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
17:20:01 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:20:01 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Curre

  10/10 Loading: s11664-021-09330-8.pdf...
Analyzing with local LLM...


17:20:09 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:20:09 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:20:09 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
17:20:09 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/lla

Direct Rb3BiI6 Synthesis completed!

DIRECT Rb3BiI6 SYNTHESIS RESULTS
The synthesis of Rb3BiI6 thin films involves the use of primary precursors such as silver(I) iodide, bismuth(III) iodide, and bismuth(III) tris(4-methylbenzodithioate) [Bi(S2CAr)3]. The molar ratio of the precursors is not explicitly stated, but the use of a 3:1 vol mixture of DMSO:DMF with 12 vol.% HI added for improved solubility suggests a specific ratio. The solution is then spin-coated and annealed at 130°C for 15 min to form the required film.

Processing parameters include spin-coating speeds and times, with a spin speed of 3000 rpm to 6000 rpm, and spin time not specified. Substrate temperatures are not mentioned, but annealing temperatures range from 150 to 320°C. The annealing temperatures were maintained without intentional substrate heating, and the films were subjected to postdeposition annealing under BiI3 vapor.

Film properties include film uniformity, morphology, thickness, and structural and optical

In [ ]:
import pandas as pd
# Get all PDF files in the directory
pdf_files = [f for f in os.listdir(pdf_dir) if f.endswith('.pdf')]
print(f"Found {len(pdf_files)} PDF files to analyze")

# Store results for each PDF
llama_all_pdf_results = {}

# Process each PDF individually
for i, pdf_file in enumerate(pdf_files, 1):
    pdf_path = os.path.join(pdf_dir, pdf_file)
    print(f"\n{'='*60}")
    print(f"ANALYZING PDF {i}/{len(pdf_files)}: {pdf_file}")
    print(f"{'='*60}")
    
    # Create temporary directory with just this PDF
    temp_dir = os.path.join(pdf_dir, f"temp_{i}")
    os.makedirs(temp_dir, exist_ok=True)
    
    # Copy single PDF to temp directory (or use symlink)
    import shutil
    temp_pdf_path = os.path.join(temp_dir, pdf_file)
    shutil.copy2(pdf_path, temp_pdf_path)
    
    try:
        # Execute extraction on single PDF
        pdf_result = run_local_analysis(
            direct_synthesis_query, 
            temp_dir, 
            f"{TARGET_COMPOSITION}_PDF_{i}_{pdf_file.replace('.pdf', '')}"
        )
        
        # Store result
        llama_all_pdf_results[pdf_file] = pdf_result
        
        # Display results for this PDF
        if pdf_result:
            # Extract answer
            if hasattr(pdf_result, 'answer'):
                answer_text = pdf_result.answer
            elif hasattr(pdf_result, 'formatted_answer'):
                answer_text = pdf_result.formatted_answer
            else:
                answer_text = str(pdf_result)
            
            print(f"\n📄 RESULTS FROM {pdf_file}:")
            print(answer_text[:500] + "..." if len(str(answer_text)) > 500 else answer_text)
            
            # Show evidence summary
            if hasattr(pdf_result, 'contexts') and pdf_result.contexts:
                print(f"\nEvidence: {len(pdf_result.contexts)} relevant passages found")
            
            print(f"✅ {pdf_file} analysis complete!")
        else:
            print(f"❌ {pdf_file} analysis failed")
            
    except Exception as e:
        print(f"❌ Error processing {pdf_file}: {str(e)}")
        llama_all_pdf_results[pdf_file] = None

    finally:
        # Clean up temp directory
        shutil.rmtree(temp_dir, ignore_errors=True)

# Summary of all results
print(f"\n{'='*60}")
print(f"SUMMARY: {TARGET_COMPOSITION} ANALYSIS COMPLETE")
print(f"{'='*60}")

llama_successful_analyses = sum(1 for result in llama_all_pdf_results.values() if result is not None)
print(f"📊 Successfully analyzed: {llama_successful_analyses}/{len(pdf_files)} PDFs")

for pdf_file, result in llama_all_pdf_results.items():
    status = "✅ Success" if result else "❌ Failed"
    print(f"   {pdf_file}: {status}")

# Save individual results to separate JSON files
llama_results_dir = os.path.join(pdf_dir, "individual_results")
os.makedirs(llama_results_dir, exist_ok=True)

# Prepare combined results structure
combined_results = {
    'analysis_metadata': {
        'target_composition': TARGET_COMPOSITION,
        'analysis_date': str(pd.Timestamp.now()),
        'total_pdfs_processed': len(pdf_files),
        'successful_analyses': len([r for r in llama_all_pdf_results.values() if r is not None]),
        'pdf_directory': pdf_dir
    },
    'individual_pdf_results': {},
    'synthesis_parameters_summary': {}
}

for pdf_file, result in llama_all_pdf_results.items():
    if result:
        # Save individual files (your existing code)
        result_file = os.path.join(llama_results_dir, f"{pdf_file.replace('.pdf', '')}_analysis.json")
        try:
            # Convert result to JSON-serializable format
            if hasattr(result, 'dict'):
                result_data = result.dict()
            elif isinstance(result, dict):
                result_data = result
            else:
                result_data = {'analysis': str(result), 'pdf_source': pdf_file}
            
            # Save individual file
            with open(result_file, 'w') as f:
                json.dump(result_data, f, indent=2)
            print(f"💾 Saved individual: {result_file}")
            
            # Add to combined results
            combined_results['individual_pdf_results'][pdf_file] = result_data
            
        except Exception as e:
            print(f"❌ Could not save {pdf_file} results: {str(e)}")
            # Still add failed result info to combined file
            combined_results['individual_pdf_results'][pdf_file] = {
                'error': str(e),
                'pdf_source': pdf_file,
                'status': 'failed'
            }

# Save combined results file
combined_file = os.path.join(pdf_dir, f"{TARGET_COMPOSITION}_llama_all_direct_synthesis_results.json")
try:
    with open(combined_file, 'w') as f:
        json.dump(combined_results, f, indent=2)
    print(f"\n🎉 Combined results saved: {combined_file}")
    print(f"📁 Individual results saved in: {llama_results_dir}")

    # Summary statistics
    print(f"\n📊 ANALYSIS SUMMARY:")
    print(f"   Total PDFs: {combined_results['analysis_metadata']['total_pdfs_processed']}")
    print(f"   Successful: {combined_results['analysis_metadata']['successful_analyses']}")
    print(f"   Failed: {len(pdf_files) - combined_results['analysis_metadata']['successful_analyses']}")
    
except Exception as e:
    print(f"❌ Could not save combined results: {str(e)}")

print(f"\n🎉 Individual PDF analysis complete!")



In [24]:
import pandas as pd
# Get all PDF files in the directory
pdf_files = [f for f in os.listdir(pdf_dir) if f.endswith('.pdf')]
print(f"Found {len(pdf_files)} PDF files to analyze")

# Store results for each PDF
qwen_all_pdf_results = {}

# Process each PDF individually
for i, pdf_file in enumerate(pdf_files, 1):
    pdf_path = os.path.join(pdf_dir, pdf_file)
    print(f"\n{'='*60}")
    print(f"ANALYZING PDF {i}/{len(pdf_files)}: {pdf_file}")
    print(f"{'='*60}")
    
    # Create temporary directory with just this PDF
    temp_dir = os.path.join(pdf_dir, f"temp_{i}")
    os.makedirs(temp_dir, exist_ok=True)
    
    # Copy single PDF to temp directory (or use symlink)
    import shutil
    temp_pdf_path = os.path.join(temp_dir, pdf_file)
    shutil.copy2(pdf_path, temp_pdf_path)
    
    try:
        # Execute extraction on single PDF
        pdf_result = run_qwen_local_analysis(
            direct_synthesis_query,
            temp_dir,
            f"{TARGET_COMPOSITION}_PDF_{i}_{pdf_file.replace('.pdf', '')}"
        )
        
        # Store result
        qwen_all_pdf_results[pdf_file] = pdf_result
        
        # Display results for this PDF
        if pdf_result:
            # Extract answer
            if hasattr(pdf_result, 'answer'):
                answer_text = pdf_result.answer
            elif hasattr(pdf_result, 'formatted_answer'):
                answer_text = pdf_result.formatted_answer
            else:
                answer_text = str(pdf_result)
            
            print(f"\n📄 RESULTS FROM {pdf_file}:")
            print(answer_text[:500] + "..." if len(str(answer_text)) > 500 else answer_text)
            
            # Show evidence summary
            if hasattr(pdf_result, 'contexts') and pdf_result.contexts:
                print(f"\nEvidence: {len(pdf_result.contexts)} relevant passages found")
            
            print(f"✅ {pdf_file} analysis complete!")
        else:
            print(f"❌ {pdf_file} analysis failed")
            
    except Exception as e:
        print(f"❌ Error processing {pdf_file}: {str(e)}")
        qwen_all_pdf_results[pdf_file] = None

    finally:
        # Clean up temp directory
        shutil.rmtree(temp_dir, ignore_errors=True)

# Summary of all results
print(f"\n{'='*60}")
print(f"SUMMARY: {TARGET_COMPOSITION} ANALYSIS COMPLETE")
print(f"{'='*60}")

qwen_successful_analyses = sum(1 for result in qwen_all_pdf_results.values() if result is not None)
print(f"📊 Successfully analyzed: {qwen_successful_analyses}/{len(pdf_files)} PDFs")

for pdf_file, result in qwen_all_pdf_results.items():
    status = "✅ Success" if result else "❌ Failed"
    print(f"   {pdf_file}: {status}")

# Save individual results to separate JSON files
qwen_results_dir = os.path.join(pdf_dir, "individual_results")
os.makedirs(qwen_results_dir, exist_ok=True)

# Prepare combined results structure
combined_results = {
    'analysis_metadata': {
        'target_composition': TARGET_COMPOSITION,
        'analysis_date': str(pd.Timestamp.now()),
        'total_pdfs_processed': len(pdf_files),
        'successful_analyses': len([r for r in qwen_all_pdf_results.values() if r is not None]),
        'pdf_directory': pdf_dir
    },
    'individual_pdf_results': {},
    'synthesis_parameters_summary': {}
}

for pdf_file, result in qwen_all_pdf_results.items():
    if result:
        # Save individual files (your existing code)
        result_file = os.path.join(qwen_results_dir, f"{pdf_file.replace('.pdf', '')}_analysis.json")
        try:
            # Convert result to JSON-serializable format
            if hasattr(result, 'dict'):
                result_data = result.dict()
            elif isinstance(result, dict):
                result_data = result
            else:
                result_data = {'analysis': str(result), 'pdf_source': pdf_file}
            
            # Save individual file
            with open(result_file, 'w') as f:
                json.dump(result_data, f, indent=2)
            print(f"💾 Saved individual: {result_file}")
            
            # Add to combined results
            combined_results['individual_pdf_results'][pdf_file] = result_data
            
        except Exception as e:
            print(f"❌ Could not save {pdf_file} results: {str(e)}")
            # Still add failed result info to combined file
            combined_results['individual_pdf_results'][pdf_file] = {
                'error': str(e),
                'pdf_source': pdf_file,
                'status': 'failed'
            }

# Save combined results file
combined_file = os.path.join(pdf_dir, f"{TARGET_COMPOSITION}_qwen_all_direct_synthesis_results.json")
try:
    with open(combined_file, 'w') as f:
        json.dump(combined_results, f, indent=2)
    print(f"\n🎉 Combined results saved: {combined_file}")
    print(f"📁 Individual results saved in: {qwen_results_dir}")

    # Summary statistics
    print(f"\n📊 ANALYSIS SUMMARY:")
    print(f"   Total PDFs: {combined_results['analysis_metadata']['total_pdfs_processed']}")
    print(f"   Successful: {combined_results['analysis_metadata']['successful_analyses']}")
    print(f"   Failed: {len(pdf_files) - combined_results['analysis_metadata']['successful_analyses']}")
    
except Exception as e:
    print(f"❌ Could not save combined results: {str(e)}")

print(f"\n🎉 Individual PDF analysis complete!")



Found 10 PDF files to analyze

ANALYZING PDF 1/10: ko1280_f.pdf
Starting: Rb3BiI6_PDF_1_ko1280_f
Analyzing papers in: /Users/shengfang/Desktop/TRI/Rb3BiI6/pdfs/temp_1
Processing 1 research papers...
  1/1 Loading: ko1280_f.pdf...


18:20:30 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
18:20:30 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
18:20:30 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
18:20:30 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30


Analyzing with local LLM...


18:22:10 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
18:22:10 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
18:22:10 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
18:22:10 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find c

Rb3BiI6_PDF_1_ko1280_f completed!

📄 RESULTS FROM ko1280_f.pdf:
I cannot answer. (Achoi2023 pages 11-12, Achoi2023 pages 53-55, Achoi2023 pages 51-52, Achoi2023 pages 5-5, Achoi2023 pages 80-81)

Evidence: 10 relevant passages found
✅ ko1280_f.pdf analysis complete!

ANALYZING PDF 2/10: Adv Funct Materials - 2022 - Chakraborty - Rudorffites and Beyond  Perovskite%E2%80%90Inspired Silver Copper Pnictohalides for-2.pdf
Starting: Rb3BiI6_PDF_2_Adv Funct Materials - 2022 - Chakraborty - Rudorffites and Beyond  Perovskite%E2%80%90Inspired Silver Copper Pnictohalides for-2
Analyzing papers in: /Users/shengfang/Desktop/TRI/Rb3BiI6/pdfs/temp_2
Processing 1 research papers...
  1/1 Loading: Adv Funct Materials - 2022 - Chakraborty - Rudorff...


...

Rb3BiI6_PDF_9_786773 completed!

📄 RESULTS FROM 786773.pdf:
I cannot answer. The provided context does not contain specific synthesis details for Rb3BiI6 thin films, including precursors, processing parameters, or film properties (Sugathan2021 pages 3-3, Sugathan2021 pages 2-2, Sugathan2021 pages 1-1).

Evidence: 10 relevant passages found
✅ 786773.pdf analysis complete!

ANALYZING PDF 10/10: s11664-021-09330-8.pdf
Starting: Rb3BiI6_PDF_10_s11664-021-09330-8
Analyzing papers in: /Users/shengfang/Desktop/TRI/Rb3BiI6/pdfs/temp_10
Processing 1 research papers...
  1/1 Loading: s11664-021-09330-8.pdf...


19:05:47 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
19:05:47 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
19:05:47 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
19:05:47 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30


Analyzing with local LLM...


19:06:22 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
19:06:22 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
19:06:22 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
19:06:22 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find c

Rb3BiI6_PDF_10_s11664-021-09330-8 completed!

📄 RESULTS FROM s11664-021-09330-8.pdf:
I cannot answer. The provided context does not contain specific synthesis details for Rb3BiI6 thin films, including precursors, processing parameters, or film properties (Achoi2022 pages 1-2, Achoi2022 pages 3-4, Achoi2022 pages 2-3, Achoi2022 pages 1-1, Achoi2022 pages 4-4).

Evidence: 10 relevant passages found
✅ s11664-021-09330-8.pdf analysis complete!

SUMMARY: Rb3BiI6 ANALYSIS COMPLETE
📊 Successfully analyzed: 10/10 PDFs
   ko1280_f.pdf: ✅ Success
   Adv Funct Materials - 2022 - Chakraborty - Rudorffites and Beyond  Perovskite%E2%80%90Inspired Silver Copper Pnictohalides for-2.pdf: ✅ Success
   The_Synthesis,_and_Structural_.pdf: ✅ Success
   a-versatile-thin-film-deposition-method-for-multidimensional-semiconducting-bismuth-halides.pdf: ✅ Success
   pubs_rsc_org_en_content_articlehtml_2022_ra_d2ra05484_2.pdf: ✅ Success
   solvent-engineering-method-to-deposit-compact-bismuth-based-thin-films-mec

In [34]:
# Query for related compound synthesis infomation
related_compounds_query = """
Extract synthesis insights for related A3BiI6 compounds:

PRECURSORS:
- Common solute chemicals across different compositions
- Molar ratios and concentrations that work for different compositions
- Solvent preferences for different precursors (DMF, DMSO, others)
- Solvent ratios for mixed systems
- General dissolution conditions (temperature, time)

PROCESSING PARAMETERS RANGES TYPICAL FOR A3BiI6 FAMILY:
- Spin-coating speeds and times
- Spin-coating steps (one-step vs multi-step, with or without antisolvent)
- Substrate temperatures (with or without preheating)
- Annealing temperatures and durations
- Atmosphere requirements (N2, air, vacuum)

PERFORMANCE CORRELATIONS:
- How synthesis conditions affect film quality including morphology, uniformity, thickness
- Common challenges and solutions

TRANSFERABLE KNOWLEDGE:
- Which parameters can be adapted from related compounds
- Best practices that apply broadly

Focus on synthesis wisdom that could guide Rb3BiI6 optimization.
"""

In [35]:
pdf_dir="/Users/shengfang/Desktop/TRI/Rb3BiI6/pdfs"
llama_related_answer = run_local_analysis(related_compounds_query, pdf_dir, "Related Rb3BiI6 Synthesis")
if llama_related_answer:
    print("\n" + "="*60)
    print("RELATED COMPOUND SYNTHESIS INSIGHTS")
    print("="*60)
    
    # Extract answer
    if hasattr(llama_related_answer, 'answer'):
        answer_text = llama_related_answer.answer
    elif hasattr(llama_related_answer, 'formatted_answer'):
        answer_text = llama_related_answer.formatted_answer
    else:
        answer_text = str(llama_related_answer)
    
    print(answer_text)
    
    # Show evidence summary
    if hasattr(llama_related_answer, 'contexts') and llama_related_answer.contexts:
        print(f"\n Evidence: {len(llama_related_answer.contexts)} relevant passages found")
    else:
        print("\n No comparative evidence found")
    
    print("\n✅ Related compounds analysis complete!")
else:
    print("❌ Related compounds analysis failed")

Starting: Related Rb3BiI6 Synthesis
Analyzing papers in: /Users/shengfang/Desktop/TRI/Rb3BiI6/pdfs
Processing 10 research papers...
  1/10 Loading: ko1280_f.pdf...


13:47:35 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
13:47:35 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
13:47:35 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
13:47:35 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30


  2/10 Loading: Adv Funct Materials - 2022 - Chakraborty - Rudorff...


13:48:23 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
13:48:23 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
13:48:23 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
13:48:23 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30


  3/10 Loading: The_Synthesis,_and_Structural_.pdf...


...

  9/10 Loading: 786773.pdf...


13:51:56 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
13:51:56 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
13:51:56 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
13:51:56 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
13:52:07 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
13:52:07 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Curre

  10/10 Loading: s11664-021-09330-8.pdf...
Analyzing with local LLM...


13:52:15 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
13:52:15 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
13:52:15 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
13:52:15 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/lla

Related Rb3BiI6 Synthesis completed!

RELATED COMPOUND SYNTHESIS INSIGHTS
Synthesis Insights for Related A3BiI6 Compounds

The synthesis of A3BiI6 compounds involves the use of various precursors, solvents, and processing parameters. Common solute chemicals include hydroiodic acid, lithium bis(trifluoromethylsulfonyl)imide, and DMF/DMSO mixtures. Solvent preferences for different precursors include DMF and DMSO, with DMSO being preferred due to its higher boiling point and formation of the BiI-DMSO-AgI intermediate adduct (Chakraborty2203 pages 13-14). Solvent ratios and concentrations vary depending on the composition, with DMF/DMSO mixtures being reported to improve solubility and crystallinity (Chakraborty2203 pages 14-15).

Processing parameters are crucial for film quality. Spin-coating speeds and times, substrate temperatures, and annealing temperatures are important parameters that can be optimized to achieve desirable film morphology and uniformity. One-step spin-coating with c

In [14]:
pdf_dir="/Users/shengfang/Desktop/TRI/Rb3BiI6/pdfs"
qwen_related_answer = run_qwen_local_analysis(related_compounds_query, pdf_dir, "Related Rb3BiI6 Synthesis")
if qwen_related_answer:
    print("\n" + "="*60)
    print("RELATED COMPOUND SYNTHESIS INSIGHTS")
    print("="*60)
    
    # Extract answer
    if hasattr(qwen_related_answer, 'answer'):
        answer_text = qwen_related_answer.answer
    elif hasattr(qwen_related_answer, 'formatted_answer'):
        answer_text = qwen_related_answer.formatted_answer
    else:
        answer_text = str(qwen_related_answer)
    
    print(answer_text)
    
    # Show evidence summary
    if hasattr(qwen_related_answer, 'contexts') and qwen_related_answer.contexts:
        print(f"\n Evidence: {len(qwen_related_answer.contexts)} relevant passages found")
    else:
        print("\n No comparative evidence found")
    
    print("\n✅ Related compounds analysis complete!")
else:
    print("❌ Related compounds analysis failed")

Starting: Related Rb3BiI6 Synthesis
Analyzing papers in: /Users/shengfang/Desktop/TRI/Rb3BiI6/pdfs
Processing 10 research papers...
  1/10 Loading: ko1280_f.pdf...
  2/10 Loading: Adv Funct Materials - 2022 - Chakraborty - Rudorff...
  3/10 Loading: The_Synthesis,_and_Structural_.pdf...
  4/10 Loading: a-versatile-thin-film-deposition-method-for-multid...
  5/10 Loading: pubs_rsc_org_en_content_articlehtml_2022_ra_d2ra05...
  6/10 Loading: solvent-engineering-method-to-deposit-compact-bism...
  7/10 Loading: ko1355_f.pdf...


Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 55 0 (offset 0)


  8/10 Loading: 786774.pdf...


Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 55 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)


  9/10 Loading: 786773.pdf...


Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)


  10/10 Loading: s11664-021-09330-8.pdf...
Analyzing with local LLM...


Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.


Related Rb3BiI6 Synthesis completed!

RELATED COMPOUND SYNTHESIS INSIGHTS
### Synthesis Insights for Related A3BiI6 Compounds

#### Precursors
Common solute chemicals across different compositions include methylammonium iodide (MAI) and bismuth iodide (BiI3) for methylammonium bismuth iodide (MBI) perovskite solar cells, and silver iodide (AgI) and bismuth iodide (BiI3) for silver bismuth iodides (AgBiI3) (Citation2022 pages 13-13, Achoi2023 pages 5-5). Molar ratios and concentrations vary, but MAI molar ratios significantly influence the morphology and performance of MBI perovskite solar cells (Achoi2023 pages 5-5). Solvent preferences include DMSO and DMF, with DMSO reducing phase segregation and pinhole formation when used in a 1:1 volume mixture with DMF (Citation2022 pages 13-14). General dissolution conditions involve temperatures of 70-110°C for less than 30 minutes to enhance solubility (Citation2022 pages 13-13).

#### Processing Parameters
Spin-coating speeds and times are cr

In [25]:
# Query for detailed element-based analysis
element_base_analysis_query = """
Extract detailed synthesis information about precursors and processing parameters for compounds containing Rubidium(Rb), Bismuth(Bi), or Iodine(I):

PRECURSOR:
- Common source chemicals for Rb, Bi, I
- Common solvents or mixed solvent systems and ratios used for these solutes
- Precursor dissolving conditions and concentrations (stirring and heating requirements, concentrations optimization ranges)
- Precursor stability and storage conditions
- Purity requirements and impurity effects
- Order of addition effects

PROCESSING PARAMETERS RANGES TYPICAL FOR COMPOUNDS CONTAINING Rb, Bi, I RESPECTIVELY:
- Spin-coating speeds and times
- Spin-coating steps (one-step vs multi-step, with or without antisolvent)
- Substrate temperatures (with or without preheating)
- Annealing temperatures and durations
- Atmosphere requirements (N2, air, vacuum)

TRANSFERABLE KNOWLEDGE:
- Which parameters can be adapted from compounds containing Rb, Bi, I
- Best practices that apply broadly

Provide specific numerical data and practical guidelines.
"""

In [26]:
# Execute analysis
llama_element_base_answer = run_local_analysis(element_base_analysis_query, pdf_dir, "Element-Based Analysis")

# Display results
if llama_element_base_answer:
    print("\n" + "="*60)
    print("ELEMENT-BASED ANALYSIS")
    print("="*60)
    
    # Extract answer
    if hasattr(llama_element_base_answer, 'answer'):
        answer_text = llama_element_base_answer.answer
    elif hasattr(llama_element_base_answer, 'formatted_answer'):
        answer_text = llama_element_base_answer.formatted_answer
    else:
        answer_text = str(llama_element_base_answer)

    print(answer_text)
    
    # Show evidence summary
    if hasattr(llama_element_base_answer, 'contexts') and llama_element_base_answer.contexts:
        print(f"\n Evidence: {len(llama_element_base_answer.contexts)} relevant passages found")
    else:
        print("\n No element-based evidence found")

    print("\n✅ Element-based analysis complete!")
else:
    print("❌ Element-based analysis failed")

Starting: Element-Based Analysis
Analyzing papers in: /Users/shengfang/Desktop/TRI/Rb3BiI6/pdfs
Processing 10 research papers...
  1/10 Loading: ko1280_f.pdf...


21:48:16 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
21:48:16 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
21:48:16 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
21:48:16 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30


  2/10 Loading: Adv Funct Materials - 2022 - Chakraborty - Rudorff...


21:49:06 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
21:49:06 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
21:49:06 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
21:49:06 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30


  3/10 Loading: The_Synthesis,_and_Structural_.pdf...


...

  9/10 Loading: 786773.pdf...


21:52:37 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
21:52:37 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
21:52:37 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
21:52:37 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
21:52:47 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
21:52:47 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Curre

  10/10 Loading: s11664-021-09330-8.pdf...
Analyzing with local LLM...


21:52:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
21:52:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
21:52:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
21:52:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/lla

Element-Based Analysis completed!

ELEMENT-BASED ANALYSIS
The synthesis of compounds containing Rubidium (Rb), Bismuth (Bi), and Iodine (I) requires careful consideration of precursor stability and storage conditions, as well as processing parameters. Common source chemicals for Rb, Bi, and I include RbI, BiI3, and I2, respectively.

The use of solvents like n-butylamine, DMSO, DMF, and their mixtures is common for dissolving precursors (Khazaee2026 pages 4-4). Precursor concentrations are typically optimized between 0.1 to 1.0 M, with stirring and heating requirements specified (Khazaee2026 pages 4-4). Precursor stability and storage conditions are crucial, with recommended storage temperatures below 10°C and purity ranges of 99.99% (Chakraborty2203 pages 19-20).

The order of addition effects are also important, with a recommended ratio of 1:1:1 for RbI:BiI3:Bi(S2CAr)3 (Chakraborty2203 pages 19-20). Purity requirements are crucial to avoid impurities, and the use of antisolvents and 

In [31]:
# Execute analysis
qwen_element_base_answer = run_qwen_local_analysis(element_base_analysis_query, pdf_dir, "Element-Based Analysis")

# Display results
if qwen_element_base_answer:
    print("\n" + "="*60)
    print("ELEMENT-BASED ANALYSIS")
    print("="*60)
    
    # Extract answer
    if hasattr(qwen_element_base_answer, 'answer'):
        answer_text = qwen_element_base_answer.answer
    elif hasattr(qwen_element_base_answer, 'formatted_answer'):
        answer_text = qwen_element_base_answer.formatted_answer
    else:
        answer_text = str(qwen_element_base_answer)

    print(answer_text)
    
    # Show evidence summary
    if hasattr(qwen_element_base_answer, 'contexts') and qwen_element_base_answer.contexts:
        print(f"\n Evidence: {len(qwen_element_base_answer.contexts)} relevant passages found")
    else:
        print("\n No element-based evidence found")

    print("\n✅ Element-based analysis complete!")
else:
    print("❌ Element-based analysis failed")

Starting: Element-Based Analysis
Analyzing papers in: /Users/shengfang/Desktop/TRI/Rb3BiI6/pdfs
Processing 10 research papers...
  1/10 Loading: ko1280_f.pdf...


22:42:42 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:42:42 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:42:42 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:42:42 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30


  2/10 Loading: Adv Funct Materials - 2022 - Chakraborty - Rudorff...


22:44:08 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:44:08 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:44:08 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:44:08 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30


  3/10 Loading: The_Synthesis,_and_Structural_.pdf...


...

  9/10 Loading: 786773.pdf...


22:50:40 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:50:40 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:50:40 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:50:40 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
22:51:10 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:51:10 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Curre

  10/10 Loading: s11664-021-09330-8.pdf...
Analyzing with local LLM...


22:51:44 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:51:44 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:51:44 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:51:44 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find c

Element-Based Analysis completed!

ELEMENT-BASED ANALYSIS
I cannot answer. The provided context does not contain detailed synthesis information about precursors and processing parameters for compounds containing Rubidium (Rb), Bismuth (Bi), or Iodine (I) as requested. The excerpts focus on silver bismuth iodide films and bismuth sulphoiodide thin films, but lack specific details on rubidium compounds or broader processing parameters for rubidium, bismuth, or iodine-containing compounds (Sugathan2021 pages 3-3, Citation2022 pages 13-13, Citation2022 pages 19-20, Sugathan2021 pages 2-2, Citation2022 pages 16-16).

 Evidence: 10 relevant passages found

✅ Element-based analysis complete!


In [32]:
# Query for other synthesis notes
other_notes_query = """
Extract other synthesis details and best practices for Rb3BiI6 and related compounds:

PROCESSING DETAILS:
- Pre-synthesis treatments
- Post-synthesis treatments
- Additive strategies

CHARACTERIZATION METHODS:
- Essential techniques for evaluating synthesis quality
- Key metrics and target values

TROUBLESHOOTING GUIDE:
- Common synthesis problems and solutions
- Film quality issues (defects, nonuniformity and impurity phases) and their causes
- Process robustness improvements

Focus on small details which are important for synthesis quality and proven best practices.
"""


In [29]:
# Execute analysis
llama_other_notes_answer = run_local_analysis(other_notes_query, pdf_dir, "Other Synthesis Notes")

# Display results
if llama_other_notes_answer:
    print("\n" + "="*60)
    print("OTHER SYNTHESIS NOTES")
    print("="*60)
    
    # Extract answer
    if hasattr(llama_other_notes_answer, 'answer'):
        answer_text = llama_other_notes_answer.answer
    elif hasattr(llama_other_notes_answer, 'formatted_answer'):
        answer_text = llama_other_notes_answer.formatted_answer
    else:
        answer_text = str(llama_other_notes_answer)

    print(answer_text)
    
    # Show evidence summary
    if hasattr(llama_other_notes_answer, 'contexts') and llama_other_notes_answer.contexts:
        print(f"\n Evidence: {len(llama_other_notes_answer.contexts)} relevant passages found")
    else:
        print("\n No other synthesis evidence found")

    print("\n✅ Other synthesis notes analysis complete!")
else:
    print("❌ Other synthesis notes analysis failed")

Starting: Other Synthesis Notes
Analyzing papers in: /Users/shengfang/Desktop/TRI/Rb3BiI6/pdfs
Processing 10 research papers...
  1/10 Loading: ko1280_f.pdf...


22:01:00 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:01:00 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:01:00 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:01:00 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30


  2/10 Loading: Adv Funct Materials - 2022 - Chakraborty - Rudorff...


22:01:44 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:01:44 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:01:44 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:01:44 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30


  3/10 Loading: The_Synthesis,_and_Structural_.pdf...


...

  9/10 Loading: 786773.pdf...


22:05:17 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:05:17 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:05:17 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:05:17 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
22:05:27 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:05:27 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Curre

  10/10 Loading: s11664-021-09330-8.pdf...
Analyzing with local LLM...


22:05:36 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:05:36 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:05:36 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:05:36 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/lla

Other Synthesis Notes completed!

OTHER SYNTHESIS NOTES
To extract other synthesis details and best practices for Rb3BiI6 and related compounds, we can analyze the provided context.

PROCESSING DETAILS:
Pre-synthesis treatments include the use of antisolvent and hot-casting methods, as well as conventional spin-coating with additives (Chakraborty2203 pages 16-16). Post-synthesis treatments include post-annealing and the use of solvents such as n-butylamine, DMSO, and DMF (Chakraborty2203 pages 16-16). Additive strategies include the use of hydroiodic acid, HCl, and DMSO to improve crystallization rates and reduce impurities (Chakraborty2203 pages 16-16).

CHARACTERIZATION METHODS:
Essential techniques for evaluating synthesis quality include evaluating synthesis quality through techniques such as X-ray diffraction and scanning electron microscopy (Chakraborty2203 pages 16-16). Key metrics and target values are not explicitly mentioned in the context.

TROUBLESHOOTING GUIDE:
Common synt

In [33]:
# Execute analysis
qwen_other_notes_answer = run_qwen_local_analysis(other_notes_query, pdf_dir, "Other Synthesis Notes")

# Display results
if qwen_other_notes_answer:
    print("\n" + "="*60)
    print("OTHER SYNTHESIS NOTES")
    print("="*60)
    
    # Extract answer
    if hasattr(qwen_other_notes_answer, 'answer'):
        answer_text = qwen_other_notes_answer.answer
    elif hasattr(qwen_other_notes_answer, 'formatted_answer'):
        answer_text = qwen_other_notes_answer.formatted_answer
    else:
        answer_text = str(qwen_other_notes_answer)

    print(answer_text)
    
    # Show evidence summary
    if hasattr(qwen_other_notes_answer, 'contexts') and qwen_other_notes_answer.contexts:
        print(f"\n Evidence: {len(qwen_other_notes_answer.contexts)} relevant passages found")
    else:
        print("\n No other synthesis evidence found")

    print("\n✅ Other synthesis notes analysis complete!")
else:
    print("❌ Other synthesis notes analysis failed")

Starting: Other Synthesis Notes
Analyzing papers in: /Users/shengfang/Desktop/TRI/Rb3BiI6/pdfs
Processing 10 research papers...
  1/10 Loading: ko1280_f.pdf...


22:56:54 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:56:54 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:56:54 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:56:54 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30


  2/10 Loading: Adv Funct Materials - 2022 - Chakraborty - Rudorff...


22:58:08 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:58:08 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:58:08 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
22:58:08 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30


  3/10 Loading: The_Synthesis,_and_Structural_.pdf...


...

  9/10 Loading: 786773.pdf...


23:04:36 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
23:04:36 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
23:04:36 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
23:04:36 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
23:05:11 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
23:05:11 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Curre

  10/10 Loading: s11664-021-09330-8.pdf...
Analyzing with local LLM...


23:05:53 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
23:05:53 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
23:05:53 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
23:05:53 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find cost for model ollama/qwen2.5:14b.
Could not find c

Other Synthesis Notes completed!

OTHER SYNTHESIS NOTES
I cannot answer.

 Evidence: 10 relevant passages found

✅ Other synthesis notes analysis complete!


In [57]:
# Get target composition from previous analysis (with fallback)
try:
    target_comp = TARGET_COMPOSITION
except NameError:
    target_comp = "Target_Composition"  # Fallback if variable not defined

print(f" Target Composition: {target_comp}")

# Collect all analysis results
llama_all_answers = []
section_titles = [
    f"Direct {target_comp} Synthesis Parameters",
    "Related Compound Synthesis Insights", 
    "Element-Based Synthesis Analysis",
    "Other Synthesis Notes"
]

# Add results from each analysis (with fallback for missing variables)
llama_results = []
try:
    if 'llama_focused_result' in locals() and llama_focused_result:
        llama_results.append(('focused_result', llama_focused_result))
except: pass

try:
    if 'llama_related_answer' in locals() and llama_related_answer:
        llama_results.append(('related_answer', llama_related_answer))
except: pass

try:
    if 'llama_element_base_answer' in locals() and llama_element_base_answer:
        llama_results.append(('element_base_answer', llama_element_base_answer))
except: pass

try:
    if 'llama_other_notes_answer' in locals() and llama_other_notes_answer:
        llama_results.append(('other_notes_answer', llama_other_notes_answer))
except: pass

# Create comprehensive summary
llama_results_summary = {
    "target_composition": target_comp,
    "source_documents": len(glob.glob(os.path.join(pdf_dir, "*.pdf"))) if 'pdf_dir' in locals() else 0,
    "pdf_directory": pdf_dir if 'pdf_dir' in locals() else "Not specified",
    "total_sections": len(section_titles),
    "sections": {}
}

print(f"Processing {len(llama_results)} analysis sections...")

# Process each result with Unicode-safe handling
for i, (var_name, answer) in enumerate(llama_results):
    title = section_titles[i] if i < len(section_titles) else f"Analysis {i+1}"
    
    try:
        # Extract answer text safely
        if hasattr(answer, 'answer'):
            raw_text = str(answer.answer)
        elif hasattr(answer, 'formatted_answer'):
            raw_text = str(answer.formatted_answer)
        else:
            raw_text = str(answer)
        
        # Clean problematic Unicode characters
        clean_text = raw_text.encode('utf-8', errors='replace').decode('utf-8', errors='replace')
        
        # Store in summary
        llama_results_summary["sections"][title] = {
            "content": clean_text,
            "evidence_count": len(answer.contexts) if hasattr(answer, 'contexts') and answer.contexts else 0,
            "analysis_variable": var_name
        }
        
        print(f"{title}: {len(clean_text)} characters")
        
    except Exception as e:
        print(f"Error processing {title}: {str(e)}")
        llama_results_summary["sections"][title] = {
            "content": f"Error processing this section: {str(e)}",
            "evidence_count": 0,
            "analysis_variable": var_name
        }

# Save comprehensive results with composition-specific filename
safe_comp_name = target_comp.replace('3', '3').replace('2', '2').lower()  # Clean filename
results_file = f"/Users/shengfang/Desktop/TRI/Rb3BiI6/{safe_comp_name}_llama_synthesis_knowledge.json"

try:
    import json
    with open(results_file, 'w', encoding='utf-8') as f:
        json.dump(llama_results_summary, f, indent=2, ensure_ascii=False)
    
    print(f"\n Comprehensive results saved to: {results_file}")
    
    # Display summary
    print("\n" + "="*70)
    print(f"COMPREHENSIVE {target_comp} SYNTHESIS KNOWLEDGE SUMMARY")
    print("="*70)
    
    print(f" Analysis Statistics:")
    print(f"   • Target Composition: {target_comp}")
    print(f"   • Source Documents: {llama_results_summary['source_documents']} PDFs")
    print(f"   • Analysis Sections: {llama_results_summary['total_sections']}")
    print(f"   • Processed Sections: {len(llama_results_summary['sections'])}")

    # Show brief section overview
    for title, data in llama_results_summary["sections"].items():
        content_preview = data["content"][:150] if data["content"] else "No content"
        print(f"\n {title}:")
        print(f"    Evidence Sources: {data['evidence_count']}")
        print(f"    Content Preview: {content_preview}...")
    
    print(f"\n✅ Complete synthesis knowledge successfully compiled!")
    print(f" Full results available in: {results_file}")
    
except Exception as e:
    print(f"❌ Error saving results: {str(e)}")

print("\n" + "="*50)
print(f" {target_comp} Analysis Complete!")
print("All synthesis knowledge has been extracted and organized.")

 Target Composition: Rb3BiI6
Processing 4 analysis sections...
Direct Rb3BiI6 Synthesis Parameters: 1112 characters
Related Compound Synthesis Insights: 1764 characters
Element-Based Synthesis Analysis: 1827 characters
Other Synthesis Notes: 1376 characters

 Comprehensive results saved to: /Users/shengfang/Desktop/TRI/Rb3BiI6/rb3bii6_llama_synthesis_knowledge.json

COMPREHENSIVE Rb3BiI6 SYNTHESIS KNOWLEDGE SUMMARY
 Analysis Statistics:
   • Target Composition: Rb3BiI6
   • Source Documents: 10 PDFs
   • Analysis Sections: 4
   • Processed Sections: 4

 Direct Rb3BiI6 Synthesis Parameters:
    Evidence Sources: 10
    Content Preview: The synthesis of Rb3BiI6 thin films involves the use of primary precursors such as silver(I) iodide, bismuth(III) iodide, and bismuth(III) tris(4-meth...

 Related Compound Synthesis Insights:
    Evidence Sources: 10
    Content Preview: Synthesis Insights for Related A3BiI6 Compounds

The synthesis of A3BiI6 compounds involves the use of various precurs

In [56]:
# Get target composition from previous analysis (with fallback)
try:
    target_comp = TARGET_COMPOSITION
except NameError:
    target_comp = "Target_Composition"  # Fallback if variable not defined

print(f" Target Composition: {target_comp}")

# Collect all analysis results
qwen_all_answers = []
section_titles = [
    f"Direct {target_comp} Synthesis Parameters",
    "Related Compound Synthesis Insights", 
    "Element-Based Synthesis Analysis",
    "Other Synthesis Notes"
]

# Add results from each analysis (with fallback for missing variables)
qwen_results = []
try:
    if 'qwen_focused_result' in locals() and qwen_focused_result:
        qwen_results.append(('focused_result', qwen_focused_result))
except: pass

try:
    if 'qwen_related_answer' in locals() and qwen_related_answer:
        qwen_results.append(('related_answer', qwen_related_answer))
except: pass

try:
    if 'qwen_element_base_answer' in locals() and qwen_element_base_answer:
        qwen_results.append(('element_base_answer', qwen_element_base_answer))
except: pass

try:
    if 'qwen_other_notes_answer' in locals() and qwen_other_notes_answer:
        qwen_results.append(('other_notes_answer', qwen_other_notes_answer))
except: pass

# Create comprehensive summary
qwen_results_summary = {
    "target_composition": target_comp,
    "source_documents": len(glob.glob(os.path.join(pdf_dir, "*.pdf"))) if 'pdf_dir' in locals() else 0,
    "pdf_directory": pdf_dir if 'pdf_dir' in locals() else "Not specified",
    "total_sections": len(section_titles),
    "sections": {}
}

print(f"Processing {len(qwen_results)} analysis sections...")

# Process each result with Unicode-safe handling
for i, (var_name, answer) in enumerate(qwen_results):
    title = section_titles[i] if i < len(section_titles) else f"Analysis {i+1}"
    
    try:
        # Extract answer text safely
        if hasattr(answer, 'answer'):
            raw_text = str(answer.answer)
        elif hasattr(answer, 'formatted_answer'):
            raw_text = str(answer.formatted_answer)
        else:
            raw_text = str(answer)
        
        # Clean problematic Unicode characters
        clean_text = raw_text.encode('utf-8', errors='replace').decode('utf-8', errors='replace')
        
        # Store in summary
        qwen_results_summary["sections"][title] = {
            "content": clean_text,
            "evidence_count": len(answer.contexts) if hasattr(answer, 'contexts') and answer.contexts else 0,
            "analysis_variable": var_name
        }
        
        print(f"{title}: {len(clean_text)} characters")
        
    except Exception as e:
        print(f"Error processing {title}: {str(e)}")
        qwen_results_summary["sections"][title] = {
            "content": f"Error processing this section: {str(e)}",
            "evidence_count": 0,
            "analysis_variable": var_name
        }

# Save comprehensive results with composition-specific filename
safe_comp_name = target_comp.replace('3', '3').replace('2', '2').lower()  # Clean filename
results_file = f"/Users/shengfang/Desktop/TRI/Rb3BiI6/{safe_comp_name}_qwen_synthesis_knowledge.json"

try:
    import json
    with open(results_file, 'w', encoding='utf-8') as f:
        json.dump(qwen_results_summary, f, indent=2, ensure_ascii=False)
    
    print(f"\n Comprehensive results saved to: {results_file}")
    
    # Display summary
    print("\n" + "="*70)
    print(f"COMPREHENSIVE {target_comp} SYNTHESIS KNOWLEDGE SUMMARY")
    print("="*70)
    
    print(f" Analysis Statistics:")
    print(f"   • Target Composition: {target_comp}")
    print(f"   • Source Documents: {qwen_results_summary['source_documents']} PDFs")
    print(f"   • Analysis Sections: {qwen_results_summary['total_sections']}")
    print(f"   • Processed Sections: {len(qwen_results_summary['sections'])}")

    # Show brief section overview
    for title, data in qwen_results_summary["sections"].items():
        content_preview = data["content"][:150] if data["content"] else "No content"
        print(f"\n {title}:")
        print(f"    Evidence Sources: {data['evidence_count']}")
        print(f"    Content Preview: {content_preview}...")
    
    print(f"\n✅ Complete synthesis knowledge successfully compiled!")
    print(f" Full results available in: {results_file}")
    
except Exception as e:
    print(f"❌ Error saving results: {str(e)}")

print("\n" + "="*50)
print(f" {target_comp} Analysis Complete!")
print("All synthesis knowledge has been extracted and organized.")

 Target Composition: Rb3BiI6
Processing 4 analysis sections...
Direct Rb3BiI6 Synthesis Parameters: 1112 characters
Related Compound Synthesis Insights: 2345 characters
Element-Based Synthesis Analysis: 560 characters
Other Synthesis Notes: 16 characters

 Comprehensive results saved to: /Users/shengfang/Desktop/TRI/Rb3BiI6/rb3bii6_qwen_synthesis_knowledge.json

COMPREHENSIVE Rb3BiI6 SYNTHESIS KNOWLEDGE SUMMARY
 Analysis Statistics:
   • Target Composition: Rb3BiI6
   • Source Documents: 10 PDFs
   • Analysis Sections: 4
   • Processed Sections: 4

 Direct Rb3BiI6 Synthesis Parameters:
    Evidence Sources: 10
    Content Preview: The synthesis of Rb3BiI6 thin films involves the use of primary precursors such as silver(I) iodide, bismuth(III) iodide, and bismuth(III) tris(4-meth...

 Related Compound Synthesis Insights:
    Evidence Sources: 10
    Content Preview: ### Synthesis Insights for Related A3BiI6 Compounds

#### Precursors
Common solute chemicals across different compositions

In [1]:
import ollama
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
def x_normalizer(X, var_array):
    
    def max_min_scaler(x, x_max, x_min):
        return (x-x_min)/(x_max-x_min)
    x_norm = []
    for x in (X):
           x_norm.append([max_min_scaler(x[i], 
                                         max(var_array[i]), 
                                         min(var_array[i])) for i in range(len(x))])
            
    return x_norm

def x_denormalizer(x_norm, var_array):
    
    def max_min_rescaler(x, x_max, x_min):
        return x*(x_max-x_min)+x_min
    x_original = []
    for x in (x_norm):
           x_original.append([max_min_rescaler(x[i], 
                                         max(var_array[i]), 
                                         min(var_array[i])) for i in range(len(x))])
            
    return x_original

def get_closest_value(given_value, array_list):
    absolute_difference_function = lambda list_value : abs(list_value - given_value)
    closest_value = min(array_list, key=absolute_difference_function)
    return closest_value
    
def get_closest_array(suggested_x, var_list):
    modified_array = []
    for x in suggested_x:
        modified_array.append([get_closest_value(x[i], var_list[i]) for i in range(len(x))])
    return np.array(modified_array)

In [5]:
def load_synthesis_knowledge(json_path):
    """Load synthesis knowledge from JSON file"""
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            knowledge = json.load(f)
        print(f"✅ Loaded synthesis knowledge for {knowledge.get('target_composition', 'Unknown')}")
        return knowledge
    except FileNotFoundError:
        print(f"❌ File not found: {json_path}")
        return None
    except json.JSONDecodeError:
        print(f"❌ Invalid JSON format in: {json_path}")
        return None

def extract_parameter_insights(knowledge, method="filtered"):
    """Extract synthesis insights from knowledge base
    
    Args:
        knowledge: Loaded JSON knowledge base
        method: "filtered" (clean format) or "raw" (full JSON as string)
    """
    
    if method == "raw":
        # Option 1: Send entire JSON as string
        return json.dumps(knowledge, indent=2)
    
    elif method == "filtered":
        # Option 2: Extract and format only synthesis content (current approach)
        insights = []
        
        if 'sections' in knowledge:
            for section_name, section_data in knowledge['sections'].items():
                content = section_data.get('content', '')
                if content and content != 'No content':
                    insights.append(f"## {section_name}\n{content}\n")
        
        return "\n".join(insights)
    
    else:
        raise ValueError("Method must be 'filtered' or 'raw'")

def generate_parameter_space_llm(synthesis_insights, target_params):
    """Use local LLM to propose parameter optimization space"""
    
    prompt = f"""
Based on the following synthesis knowledge for Rb3BiI6, propose SPECIFIC NUMERICAL RANGES AND STEP SIZES for optimization parameters.

SYNTHESIS KNOWLEDGE:
{synthesis_insights}

TARGET PARAMETERS TO OPTIMIZE:
{target_params}

Please provide SPECIFIC NUMERICAL RANGES AND STEP SIZES for each parameter based on the literature evidence:


   
1. **Spin Speed**: 
   - Range: in rpm 
   - Step Size: Increment between values 
   
2. **Precursor Concentration**: 
   - Range: in mol/L 
   - Step Size: Increment between values 
   
3. **Annealing Temperature**: 
   - Range: in °C 
   - Step Size: Increment between values

4. **Solvent Ratio**: 
   - All potential ratios of DMF:DMSO (e.g., 3:1)

5. **Solute Ratio**: 
   - All potential ratios of RbI:BiI3 (e.g., 3.2:1)

For each parameter, provide:
- Recommended minimum value
- Recommended maximum value
- Recommended step size (increment between discrete values)
- Rationale based on the synthesis knowledge
- Key literature insights that support the range and resolution

Format your response clearly with specific numbers that can be used for Bayesian optimization setup.
"""

    try:
        print("Querying local Qwen for parameter space recommendations...")
        response = ollama.chat(
            model='qwen2.5:14b',
            messages=[{
                'role': 'user', 
                'content': prompt
            }],
            options={
                'temperature': 0.0, 
                'top_k': 1.0,
                'num_predict': 2048
            }
        )
        
        return response['message']['content']
        
    except Exception as e:
        print(f"❌ Error querying LLM: {e}")
        return None

def parse_llm_recommendations(llm_output):
    """Parse LLM output to extract numerical parameter ranges"""
    
    print("\n" + "="*60)
    print("LLM PARAMETER SPACE RECOMMENDATIONS")
    print("="*60)
    print(llm_output)
    print("="*60)  
    return llm_output


# Load the synthesis knowledge JSON file
knowledge_path = "/Users/shengfang/Desktop/TRI/Rb3BiI6/rb3bii6_qwen_synthesis_knowledge.json"
synthesis_knowledge = load_synthesis_knowledge(knowledge_path)

if synthesis_knowledge:
    # Choose extraction method
    extraction_method = "filtered"  # Change to "raw" to send full JSON
    
    # Extract insights using chosen method
    insights = extract_parameter_insights(synthesis_knowledge, method=extraction_method)
    
    # Define target parameters from your optimization space
    target_parameters = """
    1. Spin Speed
    2. Precursor Concentration
    3. Annealing Temperature
    4. Solvent Ratio 
    5. Solute Ratio
    """
    
    print(f"Method: {extraction_method}")
    print(f"Analyzing {len(insights)} characters of synthesis knowledge...")

    # Generate LLM recommendations
    llm_recommendations = generate_parameter_space_llm(insights, target_parameters)
    if llm_recommendations:
        parsed_recommendations = parse_llm_recommendations(llm_recommendations)
    else:
        print("❌ No recommendations received from LLM")
        
else:
    print("❌ Could not load synthesis knowledge - using default parameter ranges")

✅ Loaded synthesis knowledge for Rb3BiI6
Method: filtered
Analyzing 4179 characters of synthesis knowledge...
Querying local Qwen for parameter space recommendations...

LLM PARAMETER SPACE RECOMMENDATIONS
Based on the provided synthesis knowledge, here are the proposed numerical ranges and step sizes for each parameter to optimize the synthesis of Rb3BiI6 thin films:

### 1. Spin Speed
- **Range**: 2500 rpm - 7000 rpm
- **Step Size**: 500 rpm

**Rationale**: The literature suggests a spin speed range from 3000 to 6000 rpm for similar compounds, but extending the range slightly lower and higher can help identify optimal conditions. A step size of 500 rpm allows for detailed exploration within this range.

### 2. Precursor Concentration
- **Range**: 0.1 mol/L - 0.8 mol/L
- **Step Size**: 0.1 mol/L

**Rationale**: The concentration significantly influences the morphology and performance of perovskite films, with higher concentrations leading to smaller crystals and potential agglomeratio

In [22]:
# Fixed parameter ranges
spinspeed_min, spinspeed_max, spinspeed_step = [2500, 5000, 500]  ## Unit: rpm
spinspeed_var = np.arange(spinspeed_min, spinspeed_max+spinspeed_step, spinspeed_step) 
spinspeed_num = len(spinspeed_var)

concentration_min, concentration_max, concentration_step = [0.1, 0.8, 0.1]  ## Unit: mol/L
concentration_var = np.arange(concentration_min, concentration_max+concentration_step, concentration_step)
concentration_num = len(concentration_var)

annealingtemp_min, annealingtemp_max, annealingtemp_step = [150, 320, 10]  # Unit: degC
annealingtemp_var = np.arange(annealingtemp_min, annealingtemp_max+annealingtemp_step, annealingtemp_step)
annealingtemp_num = len(annealingtemp_var)

# Fixed solvent and solute ratios
solvent_ratio_options = ["1:1", "2:1", "3:1"]  # String format for ratios

solute_ratio_options = ["2.8:1", "3:0:1", "3.2:1"]

In [23]:
def llm_suggest_initial_conditions(synthesis_knowledge, parameter_ranges, target_count=12):
    """
    Use LLM to suggest initial experimental conditions for Bayesian Optimization
    based on synthesis knowledge and defined parameter ranges
    """
    
    # Get synthesis insights
    insights = extract_parameter_insights(synthesis_knowledge, method="filtered")
    
    # Format parameter ranges for LLM
    param_info = f"""
PARAMETER RANGES FOR Rb3BiI6 SYNTHESIS:

1. **Spin Speed**: {spinspeed_min} - {spinspeed_max} rpm (step: {spinspeed_step} rpm)
   Available values: {list(spinspeed_var)}

2. **Precursor Concentration**: {concentration_min} - {concentration_max} mol/L (step: {concentration_step} mol/L)
   Available values: {list(concentration_var)}

3. **Annealing Temperature**: {annealingtemp_min} - {annealingtemp_max} °C (step: {annealingtemp_step} °C)
   Available values: {list(annealingtemp_var)}

4. **Solvent Ratio**: {solvent_ratio_options}

5. **Solute Ratio**: {solute_ratio_options}
"""
    
    prompt = f"""
You are an expert in synthesizing materials by spin coating. Your task is to synthesize high-quality Rb3BiI6 thin films with full coverage, pure phase and good morphology through Bayesian Optimization. Based on the following Rb3BiI6 synthesis knowledge and parameter ranges, suggest {target_count} promising experimental conditions for the initialization round of Bayesian Optimization.

SYNTHESIS KNOWLEDGE:
{insights}

{param_info}

Structure your answers in the following format:

**Sample ID**: (e.g., Sample 1, Sample 2, etc.)

-Spin Speed: (e.g., 1000 rpm)
-Precursor Concentration: (e.g., 0.5 mol/L)
-Annealing Temperature: (e.g., 200 °C)
-Solvent Ratio (DMF:DMSO): (e.g., 1:1)
-Solute Ratio (RbI:BiI3): (e.g., 3:1)
-Rationale: (e.g., This condition is promising because...)


**Guidelines for suggestion:**
- Exclude impractical parameter ranges for the condition suggestions. For example, only consider precursor concentrations that can form clear precursor solutions
- Include 4-6 conservative conditions based on provided synthesis knowledge
- Include 4-6 exploratory conditions to explore parameter space
- Include 2-4 balanced conditions between conservative and exploratory
- Ensure good coverage across all parameter dimensions after excluding impractical ranges
- Consider that these will be starting points for BO to learn from


"""

    try:
        print(f"🧠 Querying LLM for {target_count} initial BO conditions...")
        response = ollama.chat(
            model='qwen2.5:14b',
            messages=[{
                'role': 'user',
                'content': prompt
            }],
            options={
                'temperature': 0.0,  
                'top_p': 1.0,
                'num_predict': 2048  # More tokens for detailed response
            }
        )
        
        return response['message']['content']
        
    except Exception as e:
        print(f"❌ Error querying LLM for initial conditions: {e}")
        return None



In [24]:
def parse_llm_simple_and_robust(llm_response):
    """
    Simple line-by-line parser that's foolproof for your specific LLM format
    """
    import pandas as pd
    import re
    
    print("🔧 Using simple line-by-line parser...")
    
    # Split response into lines
    lines = llm_response.split('\n')
    
    samples = []
    current_sample = {}
    sample_counter = 0
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
            
        # Check for new sample
        if '**Sample ID: Sample' in line:
            # Save previous sample if complete
            if len(current_sample) >= 5:  # Has sample number + 5 parameters
                samples.append(current_sample.copy())
                print(f"✅ Completed Sample {current_sample.get('Sample', 'Unknown')}")
            
            # Start new sample
            sample_match = re.search(r'Sample\s*(\d+)', line)
            if sample_match:
                sample_counter = int(sample_match.group(1))
                current_sample = {'Sample': sample_counter}
                print(f"\n🆕 Starting Sample {sample_counter}")
            continue
        
        # Extract parameters from lines starting with "-"
        if line.startswith('- '):
            line_content = line[2:].strip()  # Remove "- " prefix
            
            # Spin Speed
            if line_content.lower().startswith('spin speed'):
                numbers = re.findall(r'(\d+)', line_content)
                if numbers:
                    current_sample['Spin_Speed_rpm'] = int(numbers[0])
                    print(f"  📊 Spin Speed: {numbers[0]} rpm")
            
            # Concentration
            elif 'concentration' in line_content.lower():
                numbers = re.findall(r'([\d.]+)', line_content)
                if numbers:
                    current_sample['Concentration_molL'] = float(numbers[0])
                    print(f"  🧪 Concentration: {numbers[0]} mol/L")
            
            # Temperature
            elif 'temperature' in line_content.lower():
                numbers = re.findall(r'(\d+)', line_content)
                if numbers:
                    current_sample['Annealing_Temp_C'] = int(numbers[0])
                    print(f"  🌡️  Temperature: {numbers[0]} °C")
            
            # Solvent Ratio - just look for any X:Y pattern
            elif 'solvent ratio' in line_content.lower():
                # Find patterns like 1:1, 2:1, etc.
                ratio_matches = re.findall(r'(\d+:\d+)', line_content)
                if ratio_matches:
                    current_sample['Solvent_Ratio'] = ratio_matches[0]
                    print(f"Solvent Ratio: {ratio_matches[0]}")
            
            # Solute Ratio - extract first number from X:Y pattern
            elif 'solute ratio' in line_content.lower():
                # Find patterns like 3:1, 2.9:1, etc. and extract first number
                ratio_match = re.search(r'(\d+:\d+)', line_content)
                if ratio_match:
                    current_sample['Solute_Ratio'] = float(ratio_match.group(1).split(':')[0])
                    print(f"Solute Ratio: {ratio_match.group(1)}")
    
    # Don't forget the last sample
    if len(current_sample) >= 5:
        samples.append(current_sample)
        print(f"✅ Completed Sample {current_sample.get('Sample', 'Unknown')}")
    
    print(f"\n📊 Total samples parsed: {len(samples)}")
    
    if samples:
        df = pd.DataFrame(samples)
        
        # Ensure all required columns exist
        required_cols = ['Sample', 'Spin_Speed_rpm', 'Concentration_molL', 'Annealing_Temp_C', 'Solvent_Ratio', 'Solute_Ratio']
        for col in required_cols:
            if col not in df.columns:
                print(f"⚠️  Missing column: {col}")
        
        return df
    else:
        return None


def test_manual_extraction(llm_response):
    """
    Manual extraction to test specific lines - for debugging
    """
    lines = llm_response.split('\n')
    
    print("🔍 Testing specific line patterns:")
    
    for i, line in enumerate(lines):
        line = line.strip()
        
        # Test solvent ratio lines
        if 'solvent ratio' in line.lower():
            print(f"Line {i}: {line}")
            ratios = re.findall(r'(\d+:\d+)', line)
            print(f"  Found ratios: {ratios}")
        
        # Test solute ratio lines  
        if 'solute ratio' in line.lower():
            print(f"Line {i}: {line}")
            ratios = re.findall(r'([\d.]+):\d+', line)
            print(f"  Found first numbers: {ratios}")

In [ ]:
# Load synthesis knowledge (use your correct path)
knowledge_path = "/Users/shengfang/Desktop/TRI/Rb3BiI6/rb3bii6_qwen_synthesis_knowledge.json"  # Update path
synthesis_knowledge = load_synthesis_knowledge(knowledge_path)

if synthesis_knowledge:
    # Get LLM suggestions for initial BO conditions
    llm_suggestions = llm_suggest_initial_conditions(
        synthesis_knowledge, 
        parameter_ranges=None,  # Ranges defined in function
        target_count=12
    )
    
    if llm_suggestions:
        print("LLM INITIAL CONDITIONS FOR BAYESIAN OPTIMIZATION:")
        print("="*100)
        print(llm_suggestions)
        print("="*100)

✅ Loaded synthesis knowledge for Rb3BiI6
🧠 Querying LLM for 12 initial BO conditions...
LLM INITIAL CONDITIONS FOR BAYESIAN OPTIMIZATION:
Based on the provided synthesis knowledge and parameter ranges, I have crafted a set of experimental conditions designed to cover both conservative and exploratory approaches while ensuring comprehensive exploration of the parameter space. Here are 12 promising initial experimental conditions:

**Sample ID**: Sample 1

- Spin Speed: 3000 rpm
- Precursor Concentration: 0.5 mol/L
- Annealing Temperature: 200 °C
- Solvent Ratio (DMF:DMSO): 1:1
- Solute Ratio (RbI:BiI3): 3:1
- Rationale: This condition is promising because it uses a balanced spin speed, moderate precursor concentration, and optimal annealing temperature based on related compound synthesis insights. The solvent ratio of 1:1 has been shown to reduce phase segregation and pinhole formation.

**Sample ID**: Sample 2

- Spin Speed: 3500 rpm
- Precursor Concentration: 0.6 mol/L
- Annealing Tem

In [26]:
output_file = "/Users/shengfang/Desktop/TRI/Rb3BiI6/llm_suggestions_raw.txt"
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(llm_suggestions)
